# Unified Pipeline v7.5.0 — When Controls Collapse

**Stimulus layer only (Phase I).** This notebook covers Cells 2 – 4.5 of the
legacy structure: configuration, word-pair generation, trial-definition
assembly, and the answer-token diagnostic. Phase A (vector collection) and
downstream RSA / patching / γ phases are deferred to a follow-up notebook.

Per Appendix A v7.5 (clean spec). Source modules under
`code/stimulus_v7_5/cells/`. Regenerate this notebook via
`uv run python build_notebook.py`.


In [ ]:
!pip install -U "transformers" "accelerate" "bitsandbytes>=0.46.1"

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
# ABSOLUTE RULE: drive.mount on line 1 of cell 1 — do not move, even for comments.
# Downstream cells block if mount is delayed to after install / imports.

# Cell 1: Install nnsight + wordfreq + rapidfuzz (no restart required)
import importlib.metadata
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'nnsight', 'wordfreq', 'rapidfuzz'])

print('nnsight:', importlib.metadata.version('nnsight'))
print('wordfreq:', importlib.metadata.version('wordfreq'))
print('rapidfuzz:', importlib.metadata.version('rapidfuzz'))
print('install OK — proceed to Cell 2')


In [ ]:
# ============================================================
# Cell 2: Config + Model Selection + HF Token
# ============================================================
# Mirror of `code/stimulus_v7_5/cells/cell2_config.py` (Appendix A v7.5).
# Edit the source-of-truth .py file and rebuild via build_notebook.py.

import os
import random
import numpy as np
import torch

# --- Pipeline identity ---
PIPELINE_VERSION = 'v7.5.0'
PROMPT_FORMAT = 'format_c_qa_options_v7_5'
TEST_MODE = False   # True => 3 pairs, small pools — smoke only

# --- HF token (carried forward per project convention; placeholder) ---
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
print('HF_TOKEN set (hardcoded placeholder).')

# --- Model selection (uncomment one per notebook copy) ---
MODEL_SPECS = {
    'llama-3.1-70b-instruct-4bit':  {'id': 'cmarkea/Meta-Llama-3.1-70B-Instruct-4bit', 'dtype': 'bfloat16'},
    'llama-3.1-8b-instruct':  {'id': 'meta-llama/Llama-3.1-8B-Instruct', 'dtype': 'bfloat16'},
    'gpt-j-6b-fp32':          {'id': 'EleutherAI/gpt-j-6B',              'dtype': 'float32'},
    'gemma-2-9b-it':          {'id': 'google/gemma-2-9b-it',             'dtype': 'bfloat16'},
    'gemma-3-27b-it':         {'id': 'google/gemma-3-27b-it',            'dtype': 'bfloat16'},
    'qwen-2.5-7b-instruct':   {'id': 'Qwen/Qwen2.5-7B-Instruct',         'dtype': 'bfloat16'},
    'qwen-2.5-14b-instruct':  {'id': 'Qwen/Qwen2.5-14B-Instruct',        'dtype': 'bfloat16'},
    'olmo-2-13b-instruct':    {'id': 'allenai/OLMo-2-1124-13B-Instruct', 'dtype': 'bfloat16'},
}
MODEL_SHORT = 'llama-3.1-70b-instruct-4bit'

MODEL_ID    = MODEL_SPECS[MODEL_SHORT]['id']
MODEL_DTYPE = MODEL_SPECS[MODEL_SHORT]['dtype']

# --- Word generation (Appendix A.1 / A.2) ---
N_PAIRS                     = 3 if TEST_MODE else 10
WORD_SEED                   = 2026
WORD_LENGTH_MIN             = 8
WORD_LENGTH_MAX             = 11
# Pool sized for ~95% success probability of finding 10 pairs under the threshold.
# 200K was overkill; 20K is >> sufficient and keeps Cell 4 under 2 minutes on A100.
CANDIDATE_POOL_SIZE         = 2000 if TEST_MODE else 20_000
MAX_ATTEMPTS                = 5000 if TEST_MODE else 200_000
COMMON_WORD_LIST_SIZE       = 3000
COMMON_WORD_LIST_URL        = 'https://pypi.org/project/wordfreq/'
LEVENSHTEIN_MIN_DISTANCE    = 3
FIRST_SUBTOKEN_MIN_LEN      = 3

LOGIT_GAP_THRESHOLDS        = (0.5, 1.0)   # primary, fallback (all-or-nothing per model)
LOGIT_GAP_NEUTRAL_CONTEXT   = 'A:'

# --- Generation (Phase A behavioural, supplementary only) ---
# Primary validity is logit-based at first answer position; string-match with
# retry ladder is recorded as supplementary metric (see Appendix A.3.3).
TEMPERATURE                 = 0.0
DO_SAMPLE                   = False
MAX_NEW_TOKENS_DEFAULT      = 50
MAX_NEW_TOKENS_RETRY        = [100, 200, 512]

# --- Attributes (Appendix A.1) ---
ATTR_P = ('bold', 'cautious')
ATTR_Q = ('complex', 'simple')

# --- Trial / position-bias grid (Appendix A.3) ---
N_POSITION_BIAS_TRIALS = 4 * 2 * N_PAIRS   # exhaustive 80

# --- Seed policy (Appendix A.4.3 — 3-module seed) ---
# ---- C.1 head batching (prompt replication, v5.1.1 proven optimization) ----
# Patch HEAD_BATCH_SIZE different heads in parallel via prompt replication.
# Speedup ~30-60x over batch=1 sequential (per v5.1.1 empirical measurement).
HEAD_BATCH_SIZE_LADDER = [128, 64, 32, 16, 8]  # OOM fallback ladder (start top)
HEAD_BATCH_SIZE = 128  # mutated by OOM handler during _run_c1

SEED = 42          # permutation / behaviour-stochastic; word_seed is 2026
PERM_SEED = SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Paths ---
LOCAL_BASE = f'/content/{MODEL_SHORT}'
DRIVE_BASE = f'/content/drive/MyDrive/WCC/models/{MODEL_SHORT}'
DRIVE_SAVE = True
PLATFORM   = 'GoogleColab_A100-SXM4-80GB'
PRINT_TIMESTAMP = True

print(f'MODEL={MODEL_SHORT}  VERSION={PIPELINE_VERSION}  TEST_MODE={TEST_MODE}')
print(f'LOCAL_BASE={LOCAL_BASE}')
print(f'DRIVE_BASE={DRIVE_BASE}')
print(f'WORD_SEED={WORD_SEED}  SEED(torch/np/random)={SEED}')
print(f'LOGIT_GAP_THRESHOLDS={LOGIT_GAP_THRESHOLDS}  neutral_context={LOGIT_GAP_NEUTRAL_CONTEXT!r}')
print(f'N_POSITION_BIAS_TRIALS={N_POSITION_BIAS_TRIALS}')


In [ ]:
# ============================================================
# Cell 3: Imports + Drive Mount + Phase-I Helpers
# ============================================================
# Consolidates logic from: common_words.py, logit_gap.py, cells/cell4_word_gen.py

import gc
import importlib.metadata
import json
import math
import re
import shutil
import time
from collections import defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime
from functools import lru_cache

import numpy as np
import pandas as pd
import transformers
import nnsight
from nnsight import LanguageModel
from wordfreq import top_n_list
from rapidfuzz.distance import Levenshtein as _RF_Levenshtein

# drive already mounted in Cell 1, line 1. Do not re-mount here.

assert torch.cuda.is_available(), 'GPU runtime required'

# --- Syllable grammar (Appendix A.1) ---
ONSETS    = ['b', 'd', 'f', 'g', 'k', 'l', 'm', 'n', 'p', 'r', 's', 't', 'v', 'w', 'z',
             'br', 'fl', 'gr', 'kr', 'pl', 'pr', 'sl', 'sp', 'st', 'tr']
VOWELS    = ['a', 'e', 'i', 'o', 'u', 'ai', 'ou', 'ei']
CODAS_MID = ['', 'n', 'l', 'r', 's']
CODAS_END = ['', 'n', 'm', 'l', 'r', 's', 't', 'k']

COMMON_WORD_LIST_SOURCE = (
    f"wordfreq.top_n_list('en', {COMMON_WORD_LIST_SIZE}, wordlist='large') "
    f"from wordfreq=={importlib.metadata.version('wordfreq')} ({COMMON_WORD_LIST_URL})"
)


# --- Logging helpers ---
def ts():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def log(msg):
    print(f'[{ts()}] {msg}' if PRINT_TIMESTAMP else msg)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return obj

def save_json(data, path, label):
    with open(path, 'w') as f:
        json.dump(to_jsonable(data), f, indent=2)
    log(f'Saved {label}: path={path}')

def count_df_nans(df):
    if not isinstance(df, pd.DataFrame) or len(df.columns) == 0:
        return 0
    return int(df.isna().sum().sum())

def save_df_csv(df, path, label):
    df_out = df if isinstance(df, pd.DataFrame) else pd.DataFrame(df)
    df_out.to_csv(path, index=False)
    log(f'Saved {label}: rows={len(df_out)} nan_count={count_df_nans(df_out)} path={path}')
    return df_out

def save_rows_csv(rows, path, label, columns=None):
    df_out = pd.DataFrame(rows, columns=columns)
    return save_df_csv(df_out, path, label)

def save_to_drive(local_dir, phase_name):
    full_drive = f'{DRIVE_BASE}/{phase_name}'
    ensure_dir(full_drive)
    copied = 0
    for root, _, files_in_root in os.walk(local_dir):
        rel_root = os.path.relpath(root, local_dir)
        dst_root = full_drive if rel_root == '.' else os.path.join(full_drive, rel_root)
        ensure_dir(dst_root)
        for fname in files_in_root:
            shutil.copy2(os.path.join(root, fname), os.path.join(dst_root, fname))
            copied += 1
    log(f'Saved to Drive: {full_drive} ({copied} files)')


# --- Phase schema check (Appendix G.3-style cache invalidation) ---
def phase_done(phase_name, logit_gap_threshold_used=None, extra_required=None):
    cfg_path = f'{DRIVE_BASE}/{phase_name}/config.json'
    if not os.path.exists(cfg_path):
        return False
    try:
        with open(cfg_path) as f:
            cfg = json.load(f)
    except Exception as exc:
        log(f'phase_done({phase_name}): failed to read config.json: {type(exc).__name__}: {exc}')
        return False
    required = {
        'pipeline_version':     PIPELINE_VERSION,
        'prompt_format':        PROMPT_FORMAT,
        'model_id':             MODEL_ID,
        'model_short':          MODEL_SHORT,
        'word_seed':            WORD_SEED,
        'n_pairs':              N_PAIRS,
        'attr_P':               list(ATTR_P),
        'attr_Q':               list(ATTR_Q),
        'common_word_list_source': COMMON_WORD_LIST_SOURCE,
    }
    if logit_gap_threshold_used is not None:
        required['logit_gap_threshold_used'] = logit_gap_threshold_used
    elif phase_name in {'10_collection', '11_position_bias'} and 'logit_gap_threshold_used' not in cfg:
        log(f'phase_done({phase_name}): missing logit_gap_threshold_used in config.json')
        return False
    if extra_required:
        required.update(extra_required)
    for k, v in required.items():
        if cfg.get(k) != v:
            log(f'phase_done({phase_name}): schema mismatch on {k} ({cfg.get(k)!r} != {v!r})')
            return False
    return True


def save_phase(local_dir, phase_name, config_dict=None):
    ensure_dir(local_dir)
    cfg_path = os.path.join(local_dir, 'config.json')
    existing_cfg = {}
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path) as fin:
                existing_cfg = json.load(fin)
        except Exception as exc:
            log(f'WARNING save_phase({phase_name}): failed to read existing config.json: {exc}')
    cfg = {**existing_cfg, **dict(config_dict or {})}
    cfg.update({
        'pipeline_version': PIPELINE_VERSION,
        'prompt_format':    PROMPT_FORMAT,
        'platform':         PLATFORM,
        'model_id':         MODEL_ID,
        'model_short':      MODEL_SHORT,
        'word_seed':        WORD_SEED,
        'perm_seed':        PERM_SEED,
        'n_pairs':          N_PAIRS,
        'attr_P':           list(ATTR_P),
        'attr_Q':           list(ATTR_Q),
        'common_word_list_source': COMMON_WORD_LIST_SOURCE,
        'saved_at':         datetime.now().isoformat(),
    })
    save_json(cfg, cfg_path, f'{phase_name}/config.json')
    if DRIVE_SAVE:
        save_to_drive(local_dir, phase_name)


# --- Model load ---
def get_torch_dtype(dtype_name):
    if dtype_name == 'float32': return torch.float32
    if dtype_name == 'float16': return torch.float16
    return torch.bfloat16

def print_vram(prefix=''):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        peak  = torch.cuda.max_memory_allocated() / 1024**3
        log(f'{prefix} VRAM allocated={alloc:.2f}GB peak={peak:.2f}GB')


def load_model(model_id, dtype_name):
    dtype = get_torch_dtype(dtype_name)
    log(f'Loading model {model_id} dtype={dtype_name}')
    model = LanguageModel(model_id, dtype=dtype, device_map='auto')
    tokenizer = model.tokenizer
    if tokenizer.pad_token_id is None and getattr(tokenizer, 'eos_token_id', None) is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    cfg = model.config
    text_cfg = getattr(cfg, 'text_config', cfg)
    num_layers = getattr(text_cfg, 'num_hidden_layers', getattr(text_cfg, 'n_layer', None))
    num_heads  = getattr(text_cfg, 'num_attention_heads', getattr(text_cfg, 'n_head', None))
    hidden     = getattr(text_cfg, 'hidden_size', getattr(text_cfg, 'n_embd', None))
    head_dim   = getattr(text_cfg, 'head_dim', None) or (hidden // num_heads if hidden and num_heads else None)
    arch = {
        'model_id':    model_id,
        'model_short': MODEL_SHORT,
        'num_layers':  num_layers,
        'num_heads':   num_heads,
        'head_dim':    head_dim,
        'hidden_size': hidden,
        'vocab_size':  getattr(text_cfg, 'vocab_size', None),
        'total_heads': (num_layers * num_heads) if num_layers and num_heads else None,
        'torch_dtype': dtype_name,
        'tuple_output': 'gemma-3' in model_id.lower(),    # matches spec naming
        'pipeline_version': PIPELINE_VERSION,
    }
    log(f'Loaded: layers={num_layers} heads={num_heads} head_dim={head_dim} vocab={arch["vocab_size"]}')
    return model, tokenizer, arch

def unload_model(model):
    try:
        del model
    except Exception as exc:
        log(f'WARNING unload_model: {exc}')
    gc.collect()
    torch.cuda.empty_cache()
    log('Model unloaded, CUDA cache cleared.')


# --- Common words / Levenshtein ---
@lru_cache(maxsize=None)
def load_common_words(n):
    raw = top_n_list('en', n, wordlist='large')
    return [w.lower() for w in raw if len(w) >= 2]

def levenshtein(s1, s2):
    # C-accelerated (rapidfuzz, 50-100x faster than pure-Python DP).
    return _RF_Levenshtein.distance(s1, s2)

def _build_common_word_buckets(common_words):
    buckets = defaultdict(list)
    for w in common_words:
        buckets[len(w)].append(w)
    return dict(buckets)

def _far_enough_from_commons(word, buckets, min_dist=LEVENSHTEIN_MIN_DISTANCE):
    # Length-bucket early exit + C-accelerated distance with score_cutoff.
    # If |len diff| >= min_dist, Levenshtein >= min_dist automatically.
    lo, hi = len(word) - (min_dist - 1), len(word) + (min_dist - 1)
    for other_len in range(lo, hi + 1):
        for cw in buckets.get(other_len, ()):
            # score_cutoff lets rapidfuzz return early (returns -1/cutoff when exceeded)
            d = _RF_Levenshtein.distance(word, cw, score_cutoff=min_dist - 1)
            if d < min_dist:
                return False
    return True

def gen_word(rng):
    syllables = []
    for i in range(3):
        coda_pool = CODAS_MID if i < 2 else CODAS_END
        syllables.append(rng.choice(ONSETS) + rng.choice(VOWELS) + rng.choice(coda_pool))
    return ''.join(syllables)


# --- Neutral-context logit cache (one forward per model) ---
def compute_neutral_logits(model):
    # nnsight 0.6: `.save()` output is a torch.Tensor directly after trace exits.
    # nnsight 0.4:  `.save()` returned a proxy and `.value` was needed to get the tensor.
    # This defensive accessor works on both (getattr returns the proxy itself if no .value).
    # Docs: https://nnsight.net/tutorials/tutorials/get_started/walkthrough/ ; v0.6 removed deprecated v0.4 API.
    with model.trace(LOGIT_GAP_NEUTRAL_CONTEXT):
        logits_proxy = model.output.logits[0, -1, :].save()
    logits_tensor = getattr(logits_proxy, 'value', logits_proxy)
    logits = logits_tensor.detach().float().cpu()
    log(f"Cached neutral logits for context {LOGIT_GAP_NEUTRAL_CONTEXT!r}: shape={tuple(logits.shape)}")
    return logits

def first_subtoken_id(tokenizer, word):
    ids = tokenizer.encode(' ' + word, add_special_tokens=False)
    if not ids:
        raise ValueError(f'Empty encoding: word={word!r}')
    return ids[0]

def prior_logit_gap(neutral_logits, tokenizer, w_a, w_b):
    t_a = first_subtoken_id(tokenizer, w_a)
    t_b = first_subtoken_id(tokenizer, w_b)
    return abs(float(neutral_logits[t_a]) - float(neutral_logits[t_b])), t_a, t_b


# --- Candidate pool (solo filters only) ---
def build_candidate_pool(common_words, seed, pool_size, max_draws=None):
    rng = random.Random(seed)
    buckets = _build_common_word_buckets(common_words)
    pool, seen = [], set()
    draws, rejects_len, rejects_lex = 0, 0, 0
    max_draws = max_draws or (pool_size * 40)
    while len(pool) < pool_size and draws < max_draws:
        draws += 1
        w = gen_word(rng).lower()
        if w in seen: continue
        if not (WORD_LENGTH_MIN <= len(w) <= WORD_LENGTH_MAX):
            rejects_len += 1
            continue
        if not _far_enough_from_commons(w, buckets):
            rejects_lex += 1
            continue
        seen.add(w); pool.append(w)
    if len(pool) < pool_size:
        raise RuntimeError(
            f'Candidate pool underfilled: {len(pool)}/{pool_size} after {draws} draws '
            f'(length-rejects={rejects_len}, lexical-rejects={rejects_lex})'
        )
    log(f'Candidate pool: {len(pool)} words in {draws} draws '
        f'(length-rejects={rejects_len}, lexical-rejects={rejects_lex})')
    return pool


# --- Pair assembly with logit-gap constraint (primary/fallback) ---
def generate_word_pairs(tokenizer, neutral_logits, candidate_pool, threshold, *, n_pairs=None, log_fn=log):
    n_pairs = n_pairs if n_pairs is not None else N_PAIRS
    rng = random.Random(WORD_SEED + 17)
    accepted, pair_gaps, used_keys = [], [], set()
    attempts = 0
    while len(accepted) < n_pairs and attempts < MAX_ATTEMPTS:
        attempts += 1
        w_a, w_b = rng.sample(candidate_pool, 2)
        if w_a[0] == w_b[0]:
            continue
        key = tuple(sorted((w_a, w_b)))
        if key in used_keys:
            continue
        ids_a = tokenizer.encode(' ' + w_a, add_special_tokens=False)
        ids_b = tokenizer.encode(' ' + w_b, add_special_tokens=False)
        if not ids_a or not ids_b: continue
        if ids_a[0] == ids_b[0]:  # first-token discriminator
            continue
        if len(ids_a) != len(ids_b):
            continue
        sub_a = tokenizer.decode([ids_a[0]]).strip()
        sub_b = tokenizer.decode([ids_b[0]]).strip()
        if len(sub_a) < FIRST_SUBTOKEN_MIN_LEN or len(sub_b) < FIRST_SUBTOKEN_MIN_LEN:
            continue
        gap, _, _ = prior_logit_gap(neutral_logits, tokenizer, w_a, w_b)
        if gap >= threshold:
            continue
        accepted.append({
            'pair_id': f'pair{len(accepted)}',
            'word_a': w_a,
            'word_b': w_b,
            'first_subtoken_id_a': ids_a[0],
            'first_subtoken_id_b': ids_b[0],
            'first_subtoken_text_a': sub_a,
            'first_subtoken_text_b': sub_b,
            'token_count_a': len(ids_a),
            'token_count_b': len(ids_b),
            'logit_gap_A_colon': gap,
        })
        pair_gaps.append({'pair_id': f'pair{len(accepted)-1}', 'word_a': w_a, 'word_b': w_b, 'gap': gap})
        used_keys.add(key)
        log_fn(f"  accepted {accepted[-1]['pair_id']}: {w_a}/{w_b} gap={gap:.4f} (threshold<{threshold})")
    if len(accepted) < n_pairs:
        return None
    return {
        'pairs': accepted,
        'pair_gaps': pair_gaps,
        'threshold_used': threshold,
        'max_gap_observed': max(r['gap'] for r in pair_gaps),
        'attempts_used': attempts,
    }


log(f'PyTorch={torch.__version__} | Transformers={transformers.__version__} | nnsight={nnsight.__version__}')
log(f'Common-word list source: {COMMON_WORD_LIST_SOURCE}')
ensure_dir(LOCAL_BASE)
ensure_dir(DRIVE_BASE)


In [ ]:
# ============================================================
# Cell 4: Word Pair Generation (Appendix A.1 / A.2)
# ============================================================

cell4_t0 = time.time()
PHASE_10_DIR = ensure_dir(os.path.join(LOCAL_BASE, '10_collection'))

# Load model once; neutral-context logits are cached for all pair candidates.
model, tokenizer, arch = load_model(MODEL_ID, MODEL_DTYPE)
save_json(arch, os.path.join(PHASE_10_DIR, 'architecture.json'), 'architecture.json')

common_words = load_common_words(COMMON_WORD_LIST_SIZE)
log(f'Loaded {len(common_words)} common words (source={COMMON_WORD_LIST_SOURCE})')

candidate_pool = build_candidate_pool(
    common_words=common_words,
    seed=WORD_SEED,
    pool_size=CANDIDATE_POOL_SIZE,
)
neutral_logits = compute_neutral_logits(model)

# Primary (< 0.5) then fallback (< 1.0) — all-or-nothing per model, no per-pair mixing.
generation_result = None
for threshold in LOGIT_GAP_THRESHOLDS:
    log(f'Attempting pair generation with threshold < {threshold}')
    result = generate_word_pairs(tokenizer, neutral_logits, candidate_pool, threshold, n_pairs=N_PAIRS)
    if result is not None:
        generation_result = result
        break
    log(f'  threshold {threshold} exhausted attempts; falling back')
if generation_result is None:
    unload_model(model)
    raise RuntimeError('Word-pair generation failed under all allowed thresholds')

accepted_pairs = generation_result['pairs']
word_pairs_df = save_rows_csv(
    accepted_pairs,
    os.path.join(PHASE_10_DIR, 'word_pairs.csv'),
    'word_pairs.csv',
    columns=['pair_id', 'word_a', 'word_b',
             'first_subtoken_id_a', 'first_subtoken_id_b',
             'first_subtoken_text_a', 'first_subtoken_text_b',
             'token_count_a', 'token_count_b',
             'logit_gap_A_colon'],
)
generation_log = {
    'pipeline_version': PIPELINE_VERSION,
    'model_id':         MODEL_ID,
    'model_short':      MODEL_SHORT,
    'prompt_format':    PROMPT_FORMAT,
    'word_seed':        WORD_SEED,
    'n_pairs':          N_PAIRS,
    'n_pairs_accepted': len(accepted_pairs),
    'candidate_pool_size': CANDIDATE_POOL_SIZE,
    'n_candidates':     len(candidate_pool),
    'attempts_cap':     MAX_ATTEMPTS,
    'attempts_used':    generation_result['attempts_used'],
    'pair_proposal_seed_offset': 17,   # rng = random.Random(WORD_SEED + 17)
    'threshold_used':   generation_result['threshold_used'],
    'thresholds_attempted': list(LOGIT_GAP_THRESHOLDS),
    'max_gap_observed': generation_result['max_gap_observed'],
    'common_word_list_source': COMMON_WORD_LIST_SOURCE,
    'common_word_list_size':   len(common_words),
    'pair_gaps':        generation_result['pair_gaps'],
}
save_json(generation_log, os.path.join(PHASE_10_DIR, 'generation_log.json'), 'generation_log.json')

cell4_elapsed_sec = time.time() - cell4_t0
save_phase(PHASE_10_DIR, '10_collection', {
    'phase_name':          '10_collection',
    'elapsed_sec':         cell4_elapsed_sec,
    'logit_gap_threshold_used': generation_result['threshold_used'],
    'max_gap_observed':    generation_result['max_gap_observed'],
    'candidate_pool_size': CANDIDATE_POOL_SIZE,
    'attempts_cap':        MAX_ATTEMPTS,
    'attempts_used':       generation_result['attempts_used'],
})
log(f"✓ word-pair generation: {len(accepted_pairs)} pairs, threshold_used={generation_result['threshold_used']}, max_gap={generation_result['max_gap_observed']:.4f}, elapsed={cell4_elapsed_sec:.1f}s")

# Keep model loaded: Cell 4.5 reuses the tokenizer; Phase A would reuse the model
# (but Phase A is out of scope for this notebook, so we free the model after the
#  stimulus layer completes).


In [ ]:
# ============================================================
# Cell 5: Trial Definitions + Position-Bias Trials (Appendix A.3)
# ============================================================

cell5_t0 = time.time()
if 'word_pairs_df' not in globals():
    raise RuntimeError('word_pairs_df not found. Run Cell 4 first.')

p_pos, p_neg = ATTR_P
q_pos, q_neg = ATTR_Q

TEMPLATE_SAMEOPP = '''Definitions:
- {word_a}: something that is {attr1} and {attr2}
- {word_b} is {relation}

Q: Is {word_b} {choice1} or {choice2}? (Answer EXACTLY ONE WORD)
A:'''

TEMPLATE_COMPARE = '''Definitions:
- {word_a}: something that is {attr1} and {attr2}
- {word_b} is {relation}

Q: Which is more {attr_asked}, {left_word} or {right_word}? (Answer EXACTLY ONE WORD)
A:'''

PHASE_11_DIR = ensure_dir(os.path.join(LOCAL_BASE, '11_position_bias'))

trials, pos_bias_trials = [], []
for row in word_pairs_df.to_dict(orient='records'):
    pid, wa, wb = row['pair_id'], row['word_a'], row['word_b']

    # Production trials
    for attr_dim, cpos, cneg in [('P', p_pos, p_neg), ('Q', q_pos, q_neg)]:
        trials.append({
            'trial_id': f'SAME_{attr_dim}_{pid}', 'crel': 'SAME', 'attr_dim': attr_dim,
            'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_SAMEOPP.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'the same as {wa}',
                                              choice1=cpos, choice2=cneg),
            'correct_answer': cpos, 'incorrect_answer': cneg, 'answer_type': 'attribute_word',
        })
        trials.append({
            'trial_id': f'OPP_{attr_dim}_{pid}', 'crel': 'OPP', 'attr_dim': attr_dim,
            'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_SAMEOPP.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'the opposite of {wa}',
                                              choice1=cpos, choice2=cneg),
            'correct_answer': cneg, 'incorrect_answer': cpos, 'answer_type': 'attribute_word',
        })
    for attr_dim, asked in [('P', p_pos), ('Q', q_pos)]:
        trials.append({
            'trial_id': f'MORE_{attr_dim}_{pid}', 'crel': 'MORE', 'attr_dim': attr_dim,
            'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_COMPARE.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'more/greater than {wa} in all respects',
                                              attr_asked=asked, left_word=wa, right_word=wb),
            'correct_answer': wb, 'incorrect_answer': wa, 'answer_type': 'novel_word',
        })
        trials.append({
            'trial_id': f'LESS_{attr_dim}_{pid}', 'crel': 'LESS', 'attr_dim': attr_dim,
            'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_COMPARE.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'less/smaller than {wa} in all respects',
                                              attr_asked=asked, left_word=wa, right_word=wb),
            'correct_answer': wa, 'incorrect_answer': wb, 'answer_type': 'novel_word',
        })

    # Position-bias: MORE/LESS = word-order swap; SAME/OPP = choice-order swap
    for attr_dim, asked in [('P', p_pos), ('Q', q_pos)]:
        pos_bias_trials.append({
            'trial_id': f'POSBIAS_MORE_{attr_dim}_{pid}', 'base_trial_id': f'MORE_{attr_dim}_{pid}',
            'crel': 'MORE', 'attr_dim': attr_dim, 'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_COMPARE.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'more/greater than {wa} in all respects',
                                              attr_asked=asked, left_word=wb, right_word=wa),
            'correct_answer': wb, 'incorrect_answer': wa, 'swap_type': 'MORE_LESS_word_swap',
        })
        pos_bias_trials.append({
            'trial_id': f'POSBIAS_LESS_{attr_dim}_{pid}', 'base_trial_id': f'LESS_{attr_dim}_{pid}',
            'crel': 'LESS', 'attr_dim': attr_dim, 'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_COMPARE.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'less/smaller than {wa} in all respects',
                                              attr_asked=asked, left_word=wb, right_word=wa),
            'correct_answer': wa, 'incorrect_answer': wb, 'swap_type': 'MORE_LESS_word_swap',
        })
    for attr_dim, cpos, cneg in [('P', p_pos, p_neg), ('Q', q_pos, q_neg)]:
        pos_bias_trials.append({
            'trial_id': f'POSBIAS_SAME_{attr_dim}_{pid}', 'base_trial_id': f'SAME_{attr_dim}_{pid}',
            'crel': 'SAME', 'attr_dim': attr_dim, 'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_SAMEOPP.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'the same as {wa}',
                                              choice1=cneg, choice2=cpos),
            'correct_answer': cpos, 'incorrect_answer': cneg, 'swap_type': 'SAMEOPP_choice_swap',
        })
        pos_bias_trials.append({
            'trial_id': f'POSBIAS_OPP_{attr_dim}_{pid}', 'base_trial_id': f'OPP_{attr_dim}_{pid}',
            'crel': 'OPP', 'attr_dim': attr_dim, 'pair_id': pid, 'word_a': wa, 'word_b': wb,
            'prompt': TEMPLATE_SAMEOPP.format(word_a=wa, word_b=wb, attr1=p_pos, attr2=q_pos,
                                              relation=f'the opposite of {wa}',
                                              choice1=cneg, choice2=cpos),
            'correct_answer': cneg, 'incorrect_answer': cpos, 'swap_type': 'SAMEOPP_choice_swap',
        })

assert len(trials) == 4 * 2 * N_PAIRS, f'Expected {4*2*N_PAIRS} production trials, got {len(trials)}'
assert len(pos_bias_trials) == N_POSITION_BIAS_TRIALS, f'Expected {N_POSITION_BIAS_TRIALS} position-bias trials, got {len(pos_bias_trials)}'

trial_definitions_df = save_rows_csv(
    trials, os.path.join(PHASE_10_DIR, 'trial_definitions.csv'), 'trial_definitions.csv',
    columns=['trial_id', 'crel', 'attr_dim', 'pair_id', 'word_a', 'word_b',
             'correct_answer', 'incorrect_answer', 'answer_type', 'prompt'],
)
position_bias_df = save_rows_csv(
    pos_bias_trials, os.path.join(PHASE_11_DIR, 'position_bias_trials.csv'), 'position_bias_trials.csv',
    columns=['trial_id', 'base_trial_id', 'crel', 'attr_dim', 'pair_id', 'word_a', 'word_b',
             'correct_answer', 'incorrect_answer', 'swap_type', 'prompt'],
)

cell5_elapsed_sec = time.time() - cell5_t0
save_phase(PHASE_10_DIR, '10_collection', {
    'phase_name': '10_collection',
    'elapsed_trial_def_sec': cell5_elapsed_sec,
    'n_trials': len(trial_definitions_df),
    'logit_gap_threshold_used': generation_log['threshold_used'],
})
save_phase(PHASE_11_DIR, '11_position_bias', {
    'phase_name': '11_position_bias',
    'elapsed_sec': cell5_elapsed_sec,
    'n_position_bias_trials': len(position_bias_df),
    'logit_gap_threshold_used': generation_log['threshold_used'],
})
log(f'✓ trial defs: {len(trial_definitions_df)} rows | position-bias: {len(position_bias_df)} rows (MORE/LESS word-swap + SAME/OPP choice-swap)')


In [ ]:
# ============================================================
# Cell 4.5: Answer-token Diagnostic (Appendix A.6)
# ============================================================
# Unified first-subtoken readout for all crels. SAME/OPP multi-token anomalies
# are warnings (not halts) per A.6.2.

cell45_t0 = time.time()
if 'trial_definitions_df' not in globals():
    raise RuntimeError('trial_definitions_df not found. Run Cell 5 first.')
if 'tokenizer' not in globals():
    raise RuntimeError('tokenizer not found. Run Cell 4 first.')

diagnostic_rows = []
warning_count = 0
first_tok_collisions = 0

for row in trial_definitions_df.to_dict(orient='records'):
    c_ids = tokenizer.encode(' ' + row['correct_answer'], add_special_tokens=False)
    i_ids = tokenizer.encode(' ' + row['incorrect_answer'], add_special_tokens=False)
    if not c_ids or not i_ids:
        raise RuntimeError(f"Empty tokenization in trial {row['trial_id']}")
    if row['crel'] in {'SAME', 'OPP'} and len(c_ids) != 1:
        warning_count += 1
        log(f"ANOMALY {row['trial_id']}: SAME/OPP correct {row['correct_answer']!r} tokenises to {c_ids} (multi-token). Logging, NOT halting.")
    if c_ids[0] == i_ids[0]:
        first_tok_collisions += 1
        log(f"COLLISION {row['trial_id']}: first_subtok(correct) == first_subtok(incorrect) == {c_ids[0]}")
    diagnostic_rows.append({
        'trial_id':                     row['trial_id'],
        'crel':                         row['crel'],
        'attr_dim':                     row['attr_dim'],
        'pair_id':                      row['pair_id'],
        'correct_answer':               row['correct_answer'],
        'incorrect_answer':             row['incorrect_answer'],
        'correct_first_subtoken_id':    c_ids[0],
        'incorrect_first_subtoken_id':  i_ids[0],
        'correct_first_subtoken_text':  tokenizer.decode([c_ids[0]]),
        'incorrect_first_subtoken_text':tokenizer.decode([i_ids[0]]),
        'correct_token_count':          len(c_ids),
        'incorrect_token_count':        len(i_ids),
    })

diagnostic_df = save_rows_csv(
    diagnostic_rows, os.path.join(PHASE_10_DIR, 'answer_token_diagnostic.csv'),
    'answer_token_diagnostic.csv',
    columns=['trial_id', 'crel', 'attr_dim', 'pair_id',
             'correct_answer', 'incorrect_answer',
             'correct_first_subtoken_id', 'incorrect_first_subtoken_id',
             'correct_first_subtoken_text', 'incorrect_first_subtoken_text',
             'correct_token_count', 'incorrect_token_count'],
)

cell45_elapsed_sec = time.time() - cell45_t0
save_phase(PHASE_10_DIR, '10_collection', {
    'phase_name': '10_collection',
    'elapsed_answer_token_diag_sec': cell45_elapsed_sec,
    'n_answer_token_diagnostic_rows': len(diagnostic_df),
    'answer_token_warning_count': warning_count,
    'answer_token_first_tok_collisions': first_tok_collisions,
    'logit_gap_threshold_used': generation_log['threshold_used'],
})
log(f'✓ diagnostic: {len(diagnostic_df)} rows, warnings={warning_count}, collisions={first_tok_collisions}, elapsed={cell45_elapsed_sec:.1f}s')

# End of Phase I. Free the model; Phase A uses a fresh load in the next notebook.
unload_model(model)
log('Phase I (stimulus layer) complete. Artefacts under 10_collection/ and 11_position_bias/.')


In [ ]:
# ============================================================
# Cell 6: Phase A — Vector Collection + Behavioural + Clean Logits (cached-aware)
# ============================================================
# Cache-aware: if artefacts + schema match on Drive, load and skip forward/generate.
# To force fresh: delete 10_collection/ and 11_position_bias/ artefacts on Drive.

phase_a_t0 = time.time()

_td_df = pd.read_csv(os.path.join(PHASE_10_DIR, 'trial_definitions.csv'))
_pb_df = pd.read_csv(os.path.join(PHASE_11_DIR, 'position_bias_trials.csv'))
_ad_df = pd.read_csv(os.path.join(PHASE_10_DIR, 'answer_token_diagnostic.csv'))
log(f'Loaded: main={len(_td_df)} trials, pos-bias={len(_pb_df)} trials, diagnostic={len(_ad_df)} rows')

PHASE_10_DRIVE_A = f'{DRIVE_BASE}/10_collection'
PHASE_11_DRIVE_A = f'{DRIVE_BASE}/11_position_bias'
PHASE_A_REQ_10 = ['activation_vectors.npz', 'clean_logits.npz', 'model_responses.csv']
PHASE_A_REQ_11 = ['position_bias_responses.csv']

def _phase_a_is_cached():
    cfg_path = f'{PHASE_10_DRIVE_A}/config.json'
    if not os.path.exists(cfg_path): return False
    try:
        with open(cfg_path) as f: cfg = json.load(f)
    except Exception: return False
    # [fix] Drop phaseA_main_elapsed_sec check: Cell 4/5/6 clobber this key on fresh VM,
    # leaving files intact but the marker missing. File-existence checks below are sufficient.
    # if cfg.get('phaseA_main_elapsed_sec') is None: return False
    for f in PHASE_A_REQ_10:
        if not os.path.exists(os.path.join(PHASE_10_DRIVE_A, f)): return False
    for f in PHASE_A_REQ_11:
        if not os.path.exists(os.path.join(PHASE_11_DRIVE_A, f)): return False
    return True

if _phase_a_is_cached():
    log('[Phase A] cached on Drive — loading artefacts (no model forward)')
    for src_dir, dst_dir in [(PHASE_10_DRIVE_A, PHASE_10_DIR), (PHASE_11_DRIVE_A, PHASE_11_DIR)]:
        for f in os.listdir(src_dir):
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    _mnpz = np.load(os.path.join(PHASE_10_DIR, 'activation_vectors.npz'))
    main_acts = {k: _mnpz[k] for k in _mnpz.files}
    _lnpz = np.load(os.path.join(PHASE_10_DIR, 'clean_logits.npz'))
    main_logits = {k: _lnpz[k] for k in _lnpz.files}
    main_rows = pd.read_csv(os.path.join(PHASE_10_DIR, 'model_responses.csv')).to_dict('records')
    pb_rows   = pd.read_csv(os.path.join(PHASE_11_DIR, 'position_bias_responses.csv')).to_dict('records')
    for r in (main_rows + pb_rows):
        for k in ('is_argmax_correct', 'is_string_correct_50', 'is_string_correct_final'):
            if k in r and isinstance(r[k], str):
                r[k] = r[k].strip().lower() in ('true', '1')
        if 'd_plus' in r and r['d_plus'] not in (None, ''):
            r['d_plus'] = float(r['d_plus'])
    log(f'  loaded {len(main_acts)} acts, {len(main_logits)} logits, {len(main_rows)} main, {len(pb_rows)} pb rows in {time.time()-phase_a_t0:.1f}s')
    # Cache-hit branch: ensure downstream phases see L/H/D/HIDDEN globals (architecture.json saved by Cell 4)
    L = int(arch['num_layers'])
    H = int(arch['num_heads'])
    D = int(arch['head_dim'])
    HIDDEN = int(arch.get('hidden_size') or (H * D))
    log(f'[Phase A cache] arch globals set: L={L} H={H} D={D} HIDDEN={HIDDEN}')
else:
    log('[Phase A] fresh run')
    model, tokenizer, arch = load_model(MODEL_ID, MODEL_DTYPE)
    L = arch['num_layers']
    H = arch['num_heads']
    D = arch['head_dim']
    assert L and H and D, f'Arch missing dims: {arch}'

    def get_decoder_layers(model):
        """Llama/Qwen/Gemma-2/OLMo — standard path."""
        core = model.model
        if hasattr(core, 'layers'):
            return core.layers
        raise AttributeError(f'Unexpected model structure for MODEL_SHORT={MODEL_SHORT}')

    layers_obj = get_decoder_layers(model)

    def _resolve_saves(saves_list):
        """Defensive accessor for nnsight 0.6 (tensor direct) + 0.4 (.value)."""
        return [getattr(p, 'value', p) for p in saves_list]

    def _first_subtoken_id(word):
        ids = tokenizer.encode(' ' + word, add_special_tokens=False)
        if not ids:
            raise ValueError(f'Empty encoding: word={word!r}')
        return ids[0]

    @torch.no_grad()
    def clean_forward(prompt, collect_activations=True):
        """v5.1.1 proven pattern on nnsight 0.6.3: accumulate per-layer slices inside
        the trace, stack them, and call .save() ONCE on the stacked tensor.

        This was validated in production on the same nnsight 0.6.3 runtime through
        2026-04-20. Rationale:
          - Only .save() return values escape the trace worker frame; plain Python
            assignments (e.g. `layer_saves = []`) do not. v5.1.1 sidesteps this by
            stacking inside and saving the stack.
          - `.cpu()` inside the trace moves per-layer slices off GPU before stack,
            keeping VRAM flat at 70B+ scale (irrelevant for 8B, but cheap insurance).
        """
        with model.trace(prompt) as tracer:
            if collect_activations:
                head_inputs = []     # worker-local; does not escape, but stack-then-save does
                for li in range(L):
                    # nnsight 0.6: .input is the first positional arg tensor; [0, -1, :]
                    # indexes batch / final position / hidden. .view(H, D) splits per-head.
                    attn = layers_obj[li].self_attn.o_proj.input[0, -1, :]
                    head_inputs.append(attn.view(H, D).cpu())
                stacked_proxy = torch.stack(head_inputs).save()   # ONE save
            logits_proxy = model.output.logits[0, -1, :].save()
        # After trace exits: .save() returns resolve to real tensors.
        logits_tensor = getattr(logits_proxy, 'value', logits_proxy)
        clean_logits = logits_tensor.detach().float().cpu().numpy()   # (vocab,)
        if collect_activations:
            stacked_tensor = getattr(stacked_proxy, 'value', stacked_proxy)
            act = stacked_tensor.detach().float().cpu().numpy()       # (L, H, D)
        else:
            act = None
        return act, clean_logits

    @torch.no_grad()
    def generate_only(prompt, max_new_tokens):
        """Separate generate block (nnsight 0.6: trace and generate do not combine)."""
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids
        input_len = int(input_ids.shape[1])
        with model.generate(prompt, max_new_tokens=max_new_tokens):
            out_proxy = model.generator.output.save()
        tokens = getattr(out_proxy, 'value', out_proxy)
        if hasattr(tokens, 'dim') and tokens.dim() == 2:
            tokens = tokens[0]
        gen_tokens = tokens[input_len:]
        return tokenizer.decode(gen_tokens, skip_special_tokens=True)

    def score_response(response, correct_answer):
        """Supplementary string-match scoring (word-boundary for attribute words)."""
        resp = response.lower().strip()
        correct = correct_answer.lower()
        attribute_words = {'bold', 'cautious', 'complex', 'simple'}
        if correct in attribute_words:
            return bool(re.search(r'\b' + re.escape(correct) + r'\b', resp))
        return correct in resp

    def run_phase_a(trials_df, diag_df, *, collect_activations, phase_label):
        """Unified Phase A loop for either main trials (activations=True) or pos-bias (False)."""
        diag_lookup = {r['trial_id']: r for r in diag_df.to_dict('records')}
        N = len(trials_df)
        results, activations, clean_logits_dict = [], {}, {}
        t0 = time.time()
        for i, trial in enumerate(trials_df.to_dict('records')):
            tid = trial['trial_id']
            prompt = trial['prompt']
            correct = trial['correct_answer']
            # Primary-validity token IDs: use the pre-computed diagnostic where possible.
            if tid in diag_lookup:
                correct_first = int(diag_lookup[tid]['correct_first_subtoken_id'])
                incorrect_first = int(diag_lookup[tid]['incorrect_first_subtoken_id'])
            else:
                # pos-bias trials: compute on-the-fly from trial row
                correct_first = _first_subtoken_id(trial['correct_answer'])
                incorrect_first = _first_subtoken_id(trial['incorrect_answer'])

            # (1) Clean forward → activations + clean logits
            act, clean_logits = clean_forward(prompt, collect_activations=collect_activations)
            if collect_activations:
                activations[tid] = act
            clean_logits_dict[tid] = clean_logits

            # (2) Primary validity — at first answer position
            d_plus = float(clean_logits[correct_first] - clean_logits[incorrect_first])
            correct_rank = int((clean_logits > clean_logits[correct_first]).sum() + 1)
            is_argmax_correct = bool(int(np.argmax(clean_logits)) == correct_first)

            # (3) Supplementary: generated-string accuracy with retry ladder
            resp_50 = generate_only(prompt, MAX_NEW_TOKENS_DEFAULT)
            is_string_50 = score_response(resp_50, correct)
            resp_final, is_string_final, max_tok_used = resp_50, is_string_50, MAX_NEW_TOKENS_DEFAULT
            if not is_string_50:
                for mnt in MAX_NEW_TOKENS_RETRY:
                    resp_try = generate_only(prompt, mnt)
                    max_tok_used = mnt
                    if score_response(resp_try, correct):
                        resp_final = resp_try
                        is_string_final = True
                        break
                    resp_final = resp_try

            row = {
                'trial_id': tid, 'crel': trial['crel'], 'attr_dim': trial['attr_dim'],
                'pair_id': trial['pair_id'],
                'word_a': trial.get('word_a'), 'word_b': trial.get('word_b'),
                'correct_answer': correct,
                'incorrect_answer': trial.get('incorrect_answer', ''),
                'correct_first_subtoken_id':   correct_first,
                'incorrect_first_subtoken_id': incorrect_first,
                'd_plus':            d_plus,
                'correct_rank':      correct_rank,
                'is_argmax_correct': is_argmax_correct,
                'generated_response_50':    resp_50[:300],
                'generated_response_final': resp_final[:300],
                'is_string_correct_50':     is_string_50,
                'is_string_correct_final':  is_string_final,
                'max_new_tokens_used':      max_tok_used,
            }
            if 'swap_type' in trial:
                row['swap_type'] = trial['swap_type']
            results.append(row)

            if (i + 1) % 10 == 0 or (i + 1) == N:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed if elapsed > 0 else 0
                eta = (N - (i + 1)) / rate if rate > 0 else 0
                log(f'  [{phase_label}] {i+1}/{N} trials  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
        return results, activations, clean_logits_dict


    # --- Main production trials: activations + clean logits + responses ---
    log('=== Phase A: main trials (activations + behavioural) ===')
    main_rows, main_acts, main_logits = run_phase_a(_td_df, _ad_df,
                                                     collect_activations=True,
                                                     phase_label='main')
    elapsed_main = time.time() - phase_a_t0

    # Save artefacts
    save_rows_csv(main_rows, os.path.join(PHASE_10_DIR, 'model_responses.csv'), 'model_responses.csv',
                  columns=['trial_id', 'crel', 'attr_dim', 'pair_id',
                           'word_a', 'word_b', 'correct_answer', 'incorrect_answer',
                           'correct_first_subtoken_id', 'incorrect_first_subtoken_id',
                           'd_plus', 'correct_rank', 'is_argmax_correct',
                           'generated_response_50', 'generated_response_final',
                           'is_string_correct_50', 'is_string_correct_final', 'max_new_tokens_used'])
    np.savez_compressed(os.path.join(PHASE_10_DIR, 'activation_vectors.npz'),
                        **{k: v for k, v in main_acts.items()})
    log(f'Saved activation_vectors.npz: {len(main_acts)} trials, shape per trial=({L},{H},{D})')
    np.savez_compressed(os.path.join(PHASE_10_DIR, 'clean_logits.npz'),
                        **{k: v for k, v in main_logits.items()})
    log(f'Saved clean_logits.npz: {len(main_logits)} trials, vocab={main_logits[list(main_logits)[0]].shape[0]}')

    # --- Position-bias trials: clean logits + responses (no activations) ---
    log('=== Phase A: position-bias trials (behavioural only) ===')
    pb_start = time.time()
    pb_rows, _pb_acts, pb_logits = run_phase_a(_pb_df, _ad_df,
                                               collect_activations=False,
                                               phase_label='posbias')
    elapsed_pb = time.time() - pb_start
    save_rows_csv(pb_rows, os.path.join(PHASE_11_DIR, 'position_bias_responses.csv'),
                  'position_bias_responses.csv',
                  columns=['trial_id', 'crel', 'attr_dim', 'pair_id',
                           'word_a', 'word_b', 'correct_answer', 'incorrect_answer',
                           'correct_first_subtoken_id', 'incorrect_first_subtoken_id',
                           'd_plus', 'correct_rank', 'is_argmax_correct',
                           'generated_response_50', 'generated_response_final',
                           'is_string_correct_50', 'is_string_correct_final',
                           'max_new_tokens_used', 'swap_type'])

    save_phase(PHASE_10_DIR, '10_collection', {
        'phase_name': '10_collection',
        'phaseA_main_elapsed_sec': elapsed_main,
        'n_main_trials': len(main_rows),
        'd_plus_rate':              sum(1 for r in main_rows if r['d_plus'] > 0) / len(main_rows),
        'argmax_accuracy':          sum(1 for r in main_rows if r['is_argmax_correct']) / len(main_rows),
        'is_string_correct_50_rate':  sum(1 for r in main_rows if r['is_string_correct_50']) / len(main_rows),
        'is_string_correct_final_rate': sum(1 for r in main_rows if r['is_string_correct_final']) / len(main_rows),
    })
    save_phase(PHASE_11_DIR, '11_position_bias', {
        'phase_name': '11_position_bias',
        'phaseA_posbias_elapsed_sec': elapsed_pb,
        'n_posbias_trials': len(pb_rows),
        'd_plus_rate_posbias':     sum(1 for r in pb_rows if r['d_plus'] > 0) / len(pb_rows),
        'argmax_accuracy_posbias': sum(1 for r in pb_rows if r['is_argmax_correct']) / len(pb_rows),
    })

    phase_a_total = time.time() - phase_a_t0
    log(f'✓ Phase A complete: main {len(main_rows)} trials + pos-bias {len(pb_rows)} trials in {phase_a_total:.0f}s')
    log(f'  main  D+ rate={100*sum(1 for r in main_rows if r["d_plus"] > 0)/len(main_rows):.1f}% | argmax-accuracy={100*sum(1 for r in main_rows if r["is_argmax_correct"])/len(main_rows):.1f}%')
    log(f'  main  string@50={100*sum(1 for r in main_rows if r["is_string_correct_50"])/len(main_rows):.1f}% | string@final={100*sum(1 for r in main_rows if r["is_string_correct_final"])/len(main_rows):.1f}%')
    log(f'  pb    D+ rate={100*sum(1 for r in pb_rows if r["d_plus"] > 0)/len(pb_rows):.1f}% | argmax-accuracy={100*sum(1 for r in pb_rows if r["is_argmax_correct"])/len(pb_rows):.1f}%')

    unload_model(model)

# --- Summary (runs for both cached and fresh paths) ---
log('=== Phase A summary ===')
_n_main = len(main_rows); _n_pb = len(pb_rows)
log(f'✓ main  n={_n_main}  D+={100*sum(1 for r in main_rows if (r["d_plus"] or 0) > 0)/_n_main:.1f}%  argmax={100*sum(1 for r in main_rows if r["is_argmax_correct"])/_n_main:.1f}%  str@50={100*sum(1 for r in main_rows if r["is_string_correct_50"])/_n_main:.1f}%  str@final={100*sum(1 for r in main_rows if r["is_string_correct_final"])/_n_main:.1f}%')
log(f'✓ pb    n={_n_pb}  D+={100*sum(1 for r in pb_rows if (r["d_plus"] or 0) > 0)/_n_pb:.1f}%  argmax={100*sum(1 for r in pb_rows if r["is_argmax_correct"])/_n_pb:.1f}%')

log('--- main: per-crel accuracy ---')
for _crel in ('SAME', 'OPP', 'MORE', 'LESS'):
    _sub = [r for r in main_rows if r['crel'] == _crel]
    if not _sub: continue
    _dp = sum(1 for r in _sub if (r['d_plus'] or 0) > 0) / len(_sub)
    _am = sum(1 for r in _sub if r['is_argmax_correct']) / len(_sub)
    _sf = sum(1 for r in _sub if r['is_string_correct_final']) / len(_sub)
    log(f'  {_crel}: n={len(_sub)}  D+={_dp*100:.1f}%  argmax={_am*100:.1f}%  str@final={_sf*100:.1f}%')

log('--- main: per-attr_dim accuracy ---')
for _attr in ('P', 'Q'):
    _sub = [r for r in main_rows if r['attr_dim'] == _attr]
    if not _sub: continue
    _dp = sum(1 for r in _sub if (r['d_plus'] or 0) > 0) / len(_sub)
    _am = sum(1 for r in _sub if r['is_argmax_correct']) / len(_sub)
    _sf = sum(1 for r in _sub if r['is_string_correct_final']) / len(_sub)
    log(f'  {_attr}: n={len(_sub)}  D+={_dp*100:.1f}%  argmax={_am*100:.1f}%  str@final={_sf*100:.1f}%')

log('--- MORE/LESS response visual check (first 10 each) ---')
for _crel in ('MORE', 'LESS'):
    for _r in [x for x in main_rows if x['crel'] == _crel][:10]:
        _flag = '✓' if (_r['d_plus'] or 0) > 0 else '✗'
        log(f'  [{_flag}] {_r["trial_id"]}: correct={_r["correct_answer"]!r}  d_plus={float(_r["d_plus"]):.3f}  resp_final={str(_r["generated_response_final"])[:60]!r}')

log(f'✓ Phase A all done in {time.time() - phase_a_t0:.0f}s')


In [ ]:
# ============================================================
# Cell 8: Phase B — RSA + Perturbation-L2 + Cell Classification
# Purpose: score Phase A activations into per-head RSA importance,
# perturbation magnitude, and 2x2 cell labels.
# Inputs already in scope: L, H, D, arch, main_acts, main_logits,
# main_rows, _td_df, log(), ensure_dir(), save_json(), save_rows_csv(),
# save_phase(), phase_done(), LOCAL_BASE, DRIVE_BASE, PIPELINE_VERSION.
# Outputs: 20_scoring/{rsa_per_head.csv, rsa_per_head_partial.csv,
# rsa_sensitivity_no_less.csv, perturbation_per_head.csv,
# cell_classification.csv, median_thresholds.json, config.json}.
# Appendix refs: A.4.1 (RSA), A.4.2 (perturbation-L2), A.5.2 (rule alpha).
# ============================================================

from scipy.stats import rankdata
import resource

PHASE_20_DIR = ensure_dir(os.path.join(LOCAL_BASE, '20_scoring'))
PHASE_20_DRIVE = os.path.join(DRIVE_BASE, '20_scoring')
N_PERMS = 10_000
CREL_PAIRS = [
    ('SAME', 'OPP'), ('SAME', 'MORE'), ('SAME', 'LESS'),
    ('OPP', 'MORE'), ('OPP', 'LESS'), ('MORE', 'LESS'),
]
PHASE_20_FILES = [
    'rsa_per_head.csv', 'rsa_per_head_partial.csv', 'rsa_sensitivity_no_less.csv',
    'perturbation_per_head.csv', 'cell_classification.csv',
    'median_thresholds.json', 'config.json',
]
expected_cfg = {
    'phase_name': '20_scoring',
    'perm_seed': PERM_SEED,
    'n_perms': N_PERMS,
    'n_trials_rsa': 80,
    'n_trials_sensitivity': 60,
    'n_layers': int(L),
    'n_heads': int(H),
    'perturbation_tuples_per_head': 120,
}


def _log_mem(tag):
    rss_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
        log(f'[{tag}] RAM(maxrss)={rss_gb:.2f}GB VRAM alloc={alloc:.2f}GB peak={peak:.2f}GB')
    else:
        log(f'[{tag}] RAM(maxrss)={rss_gb:.2f}GB CUDA=off')


def _copy_drive_to_local(src_dir, dst_dir):
    ensure_dir(dst_dir)
    for fname in os.listdir(src_dir):
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))


def _standardize(vec):
    vec = np.asarray(vec, dtype=np.float32)
    centered = vec - float(vec.mean())
    denom = float(np.sqrt(np.sum(centered * centered)))
    if denom <= 1e-12:
        return np.full_like(vec, np.nan), float(vec.mean()), denom
    return centered / denom, float(vec.mean()), denom


def _safe_partial(r_xy, r_xz, r_yz):
    # Partial-correlation identity:
    # r_xy.z = (r_xy - r_xz*r_yz) / sqrt((1-r_xz^2)(1-r_yz^2))
    # This is the standard residualized partial-correlation formula requested in B-5.
    denom = np.sqrt(np.maximum(0.0, 1.0 - r_xz * r_xz) * np.maximum(0.0, 1.0 - r_yz * r_yz))
    out = np.full_like(r_xy, np.nan, dtype=np.float32)
    mask = np.isfinite(r_xy) & np.isfinite(r_xz) & np.isfinite(denom) & (denom > 1e-12)
    out[mask] = (r_xy[mask] - r_xz[mask] * r_yz) / denom[mask]
    return np.clip(out, -1.0, 1.0)


def _quartile_labels(values):
    labels4 = ['Q1', 'Q2', 'Q3', 'Q4']
    s = pd.Series(values, dtype='float64')
    try:
        return pd.qcut(s, q=4, labels=labels4, duplicates='drop').astype('object')
    except ValueError:
        codes = pd.qcut(s, q=4, labels=False, duplicates='drop')
        if codes is None:
            return pd.Series([np.nan] * len(s), dtype='object')
        out = pd.Series([np.nan] * len(s), dtype='object')
        finite = pd.notna(codes)
        if finite.any():
            n_bins = int(codes[finite].max()) + 1
            labels = labels4[:n_bins]
            out.loc[finite] = [labels[int(c)] for c in codes[finite]]
        return out


def _compute_rsa_suite(trial_df, suite_name, compute_partial):
    t_suite = time.time()
    tids = trial_df['trial_id'].tolist()
    n_trials = len(tids)
    tri_i, tri_j = np.triu_indices(n_trials, k=1)
    n_pairs = len(tri_i)
    total_heads = L * H
    trial_acts = np.stack([main_acts[tid] for tid in tids], axis=0).astype(np.float32, copy=False)
    head_acts = trial_acts.transpose(1, 2, 0, 3).reshape(total_heads, n_trials, D)
    crel_codes = pd.Categorical(trial_df['crel']).codes.astype(np.int16)
    attr_codes = pd.Categorical(trial_df['attr_dim']).codes.astype(np.int16)
    pair_codes = pd.Categorical(trial_df['pair_id']).codes.astype(np.int16)
    crel_vec = (crel_codes[tri_i] == crel_codes[tri_j]).astype(np.float32)
    attr_vec = (attr_codes[tri_i] == attr_codes[tri_j]).astype(np.float32)
    pair_vec = (pair_codes[tri_i] == pair_codes[tri_j]).astype(np.float32)
    crel_std, crel_mean, crel_norm = _standardize(crel_vec)
    attr_std, attr_mean, attr_norm = _standardize(attr_vec)
    pair_std, _, _ = _standardize(pair_vec)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tri_i_t = torch.as_tensor(tri_i, device=device)
    tri_j_t = torch.as_tensor(tri_j, device=device)
    neural_ranked = np.zeros((total_heads, n_pairs), dtype=np.float32)
    valid = np.ones(total_heads, dtype=bool)
    bad_heads = []
    head_chunk = 256 if torch.cuda.is_available() else 64
    log(f'[{suite_name}] ranking {total_heads} heads over {n_trials} trials / {n_pairs} pairs on {device.type}')
    for start in range(0, total_heads, head_chunk):
        end = min(start + head_chunk, total_heads)
        x = torch.as_tensor(head_acts[start:end], device=device, dtype=torch.float32)
        x = x / torch.clamp(torch.linalg.vector_norm(x, dim=-1, keepdim=True), min=1e-12)
        sims = torch.matmul(x, x.transpose(1, 2))[:, tri_i_t, tri_j_t].detach().cpu().numpy()
        for off, vec in enumerate(sims):
            idx = start + off
            if not np.isfinite(vec).all() or np.nanstd(vec) <= 1e-12:
                valid[idx] = False
                bad_heads.append((idx // H, idx % H))
                continue
            ranks = rankdata(vec, method='average').astype(np.float32)
            ranks -= float(ranks.mean())
            denom = float(np.sqrt(np.sum(ranks * ranks)))
            if denom <= 1e-12:
                valid[idx] = False
                bad_heads.append((idx // H, idx % H))
            else:
                neural_ranked[idx] = ranks / denom
        if end % 200 == 0 or end == total_heads:
            log(f'[{suite_name}] ranked heads {end}/{total_heads}')
        del x, sims
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if bad_heads:
        log(f'[{suite_name}] WARNING degenerate neural RDMs: {len(bad_heads)} heads, first={bad_heads[:5]}')
    obs_crel = neural_ranked @ crel_std
    obs_attr = neural_ranked @ attr_std
    obs_pair = neural_ranked @ pair_std
    obs_crel[~valid] = np.nan
    obs_attr[~valid] = np.nan
    obs_pair[~valid] = np.nan
    partial_crel = partial_attr = None
    partial_elapsed = 0.0
    if compute_partial:
        t_partial = time.time()
        partial_crel = _safe_partial(obs_crel, obs_pair, float(np.nansum(crel_std * pair_std)))
        partial_attr = _safe_partial(obs_attr, obs_pair, float(np.nansum(attr_std * pair_std)))
        partial_crel[~valid] = np.nan
        partial_attr[~valid] = np.nan
        partial_elapsed = time.time() - t_partial
        log(f'[{suite_name}] partial RSA done in {partial_elapsed:.1f}s')
    p_crel = np.full(total_heads, np.nan, dtype=np.float32)
    p_attr = np.full(total_heads, np.nan, dtype=np.float32)
    valid_idx = np.flatnonzero(valid)
    if len(valid_idx):
        rng = np.random.default_rng(PERM_SEED)
        nr_t = torch.as_tensor(neural_ranked[valid_idx], device=device, dtype=torch.float32)
        abs_crel_t = torch.as_tensor(np.abs(obs_crel[valid_idx]), device=device, dtype=torch.float32)[:, None]
        abs_attr_t = torch.as_tensor(np.abs(obs_attr[valid_idx]), device=device, dtype=torch.float32)[:, None]
        count_crel = np.zeros(len(valid_idx), dtype=np.int64)
        count_attr = np.zeros(len(valid_idx), dtype=np.int64)
        perm_batch = 250
        t_perm = time.time()
        for p0 in range(0, N_PERMS, perm_batch):
            bs = min(perm_batch, N_PERMS - p0)
            perms = np.stack([rng.permutation(n_trials) for _ in range(bs)], axis=0)
            pc = (crel_codes[perms[:, tri_i]] == crel_codes[perms[:, tri_j]]).astype(np.float32)
            pa = (attr_codes[perms[:, tri_i]] == attr_codes[perms[:, tri_j]]).astype(np.float32)
            pc_t = torch.as_tensor(((pc - crel_mean) / crel_norm).T, device=device, dtype=torch.float32)
            pa_t = torch.as_tensor(((pa - attr_mean) / attr_norm).T, device=device, dtype=torch.float32)
            score_crel = nr_t @ pc_t
            score_attr = nr_t @ pa_t
            count_crel += (torch.abs(score_crel) >= abs_crel_t).sum(dim=1).cpu().numpy()
            count_attr += (torch.abs(score_attr) >= abs_attr_t).sum(dim=1).cpu().numpy()
            done = p0 + bs
            if done % 1000 == 0 or done == N_PERMS:
                elapsed = time.time() - t_perm
                rate = done / max(elapsed, 1e-9)
                eta = (N_PERMS - done) / max(rate, 1e-9) / 60.0
                log(f'[{suite_name}] perms {done}/{N_PERMS} elapsed={elapsed:.1f}s ETA={eta:.1f}m')
            del perms, pc, pa, pc_t, pa_t, score_crel, score_attr
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        p_crel[valid_idx] = count_crel.astype(np.float32) / float(N_PERMS)
        p_attr[valid_idx] = count_attr.astype(np.float32) / float(N_PERMS)
    rsa_pre = np.maximum(obs_crel, obs_attr)
    rsa_max = np.where(np.isfinite(rsa_pre), np.maximum(rsa_pre, 0.0), np.nan)
    raw_df = pd.DataFrame({
        'layer': np.repeat(np.arange(L), H),
        'head': np.tile(np.arange(H), L),
        'rsa_crel': obs_crel.astype(np.float32),
        'rsa_attr': obs_attr.astype(np.float32),
        'rsa_max': rsa_max.astype(np.float32),
        'rsa_max_before_clamp': rsa_pre.astype(np.float32),
        'rsa_crel_pval': p_crel.astype(np.float32),
        'rsa_attr_pval': p_attr.astype(np.float32),
    })
    partial_df = None
    if compute_partial:
        partial_max = np.where(
            np.isfinite(partial_crel) | np.isfinite(partial_attr),
            np.fmax(partial_crel, partial_attr),
            np.nan,
        )
        partial_df = pd.DataFrame({
            'layer': np.repeat(np.arange(L), H),
            'head': np.tile(np.arange(H), L),
            'rsa_crel_partial': partial_crel.astype(np.float32),
            'rsa_attr_partial': partial_attr.astype(np.float32),
            'rsa_max_partial': partial_max.astype(np.float32),
        })
    del trial_acts, head_acts, neural_ranked
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return raw_df, partial_df, max(0.0, time.time() - t_suite - partial_elapsed), partial_elapsed


def _compute_perturbation():
    t0 = time.time()
    tid_lookup = {(r['crel'], r['pair_id'], r['attr_dim']): r['trial_id'] for r in _td_df.to_dict('records')}
    clean_ids, source_ids, missing = [], [], []
    for c1, c2 in CREL_PAIRS:
        for pair_id in sorted(_td_df['pair_id'].unique()):
            for attr_dim in sorted(_td_df['attr_dim'].unique()):
                tid1 = tid_lookup.get((c1, pair_id, attr_dim))
                tid2 = tid_lookup.get((c2, pair_id, attr_dim))
                if tid1 is None or tid2 is None:
                    missing.append((c1, c2, pair_id, attr_dim))
                else:
                    clean_ids.append(tid1)
                    source_ids.append(tid2)
    if missing:
        log(f'[perturbation] WARNING missing tuples for rule alpha: {len(missing)}; first={missing[:5]}')
        pert = np.full((L, H), np.nan, dtype=np.float32)
    else:
        clean = np.stack([main_acts[tid] for tid in clean_ids], axis=0).astype(np.float32, copy=False)
        source = np.stack([main_acts[tid] for tid in source_ids], axis=0).astype(np.float32, copy=False)
        if torch.cuda.is_available():
            diff = torch.as_tensor(clean - source, device='cuda', dtype=torch.float32)
            pert = torch.linalg.vector_norm(diff, dim=-1).mean(dim=0).detach().cpu().numpy()
            del diff
            torch.cuda.empty_cache()
        else:
            pert = np.linalg.norm(clean - source, axis=-1).mean(axis=0).astype(np.float32)
        del clean, source
    pert_df = pd.DataFrame({
        'layer': np.repeat(np.arange(L), H),
        'head': np.tile(np.arange(H), L),
        'perturbation_L2': pert.reshape(-1).astype(np.float32),
    })
    gc.collect()
    return pert_df, len(clean_ids), time.time() - t0


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
_log_mem('Phase B start')
phase_b_cached = phase_done('20_scoring', extra_required=expected_cfg) and all(
    os.path.exists(os.path.join(PHASE_20_DRIVE, fname)) for fname in PHASE_20_FILES
)
if phase_done('20_scoring', extra_required=expected_cfg) and not phase_b_cached:
    log('Phase B config matches but one or more cached artefacts are missing; recomputing 20_scoring')

if phase_b_cached:
    log('[Phase B] cached on Drive — loading artefacts (no recompute)')
    _copy_drive_to_local(PHASE_20_DRIVE, PHASE_20_DIR)
    rsa_per_head_df = pd.read_csv(os.path.join(PHASE_20_DIR, 'rsa_per_head.csv'))
    perturbation_per_head_df = pd.read_csv(os.path.join(PHASE_20_DIR, 'perturbation_per_head.csv'))
    cell_classification_df = pd.read_csv(os.path.join(PHASE_20_DIR, 'cell_classification.csv'))
    globals()['rsa_per_head_df'] = rsa_per_head_df
    globals()['perturbation_per_head_df'] = perturbation_per_head_df
    globals()['cell_classification_df'] = cell_classification_df
    log(f'[Phase B] loaded cached artefacts: rsa={len(rsa_per_head_df)} pert={len(perturbation_per_head_df)} cells={len(cell_classification_df)}')
else:
    phase_b_t0 = time.time()
    log('=== Phase B: RSA + Perturbation + Cell Classification ===')
    rsa_per_head_df, rsa_partial_df, rsa_elapsed_sec, partial_rsa_elapsed_sec = _compute_rsa_suite(
        _td_df.copy(), 'rsa-80', compute_partial=True
    )
    rsa_sensitivity_no_less_df, _, sensitivity_elapsed_sec, _ = _compute_rsa_suite(
        _td_df[_td_df['crel'] != 'LESS'].reset_index(drop=True), 'rsa-60-no-less', compute_partial=False
    )
    perturbation_per_head_df, tuple_count, perturbation_elapsed_sec = _compute_perturbation()
    if tuple_count != 120:
        log(f'WARNING perturbation tuple count is {tuple_count}, expected 120')
    rsa_vals = rsa_per_head_df['rsa_max'].to_numpy(dtype=float)
    pert_vals = perturbation_per_head_df['perturbation_L2'].to_numpy(dtype=float)
    median_rsa_max = float(np.nanmedian(rsa_vals)) if np.isfinite(rsa_vals).any() else float('nan')
    median_pert = float(np.nanmedian(pert_vals)) if np.isfinite(pert_vals).any() else float('nan')
    n_tied_imp = int(np.sum(np.isfinite(rsa_vals) & (rsa_vals == median_rsa_max))) if np.isfinite(median_rsa_max) else 0
    n_tied_pert = int(np.sum(np.isfinite(pert_vals) & (pert_vals == median_pert))) if np.isfinite(median_pert) else 0
    neg_mask = np.isfinite(rsa_per_head_df['rsa_max_before_clamp']) & (rsa_per_head_df['rsa_max_before_clamp'] < 0)
    neg_vals = rsa_per_head_df.loc[neg_mask, 'rsa_max_before_clamp'].to_numpy(dtype=float)
    neg_count = int(neg_mask.sum())
    neg_min = float(np.min(neg_vals)) if neg_count else None
    neg_max = float(np.max(neg_vals)) if neg_count else None
    log(f'Negative pre-clamp heads: {neg_count}/{L*H}; range=[{neg_min}, {neg_max}] vs median rsa_max={median_rsa_max}')
    if (neg_count / float(L * H)) >= 0.05:
        log('WARNING non-negativity hypothesis may need revisit for this model')
    quartiles = _quartile_labels(rsa_vals)
    cells = []
    for i, row in rsa_per_head_df.iterrows():
        rsa_max_i = float(row['rsa_max'])
        pert_i = float(perturbation_per_head_df.iloc[i]['perturbation_L2'])
        if not (np.isfinite(rsa_max_i) and np.isfinite(pert_i) and np.isfinite(median_rsa_max) and np.isfinite(median_pert)):
            cell = np.nan
            log(f'WARNING classification NaN at layer={int(row.layer)} head={int(row.head)}')
        else:
            imp_hi = rsa_max_i >= median_rsa_max
            pert_hi = pert_i >= median_pert
            cell = 'hihp' if imp_hi and pert_hi else 'hilp' if imp_hi else 'C' if pert_hi else 'D'
        cells.append(cell)
    cell_classification_df = pd.DataFrame({
        'layer': rsa_per_head_df['layer'].astype(int),
        'head': rsa_per_head_df['head'].astype(int),
        'cell': cells,
        'rsa_max': rsa_per_head_df['rsa_max'].astype(np.float32),
        'perturbation_L2': perturbation_per_head_df['perturbation_L2'].astype(np.float32),
        'importance_quartile': quartiles,
    })
    cell_counts = {k: int((cell_classification_df['cell'] == k).sum()) for k in ['hihp', 'hilp', 'C', 'D']}
    median_thresholds = {
        'median_rsa_max': median_rsa_max,
        'median_perturbation_L2': median_pert,
        'n_tied_on_median_importance': n_tied_imp,
        'n_tied_on_median_perturbation': n_tied_pert,
        'n_heads_total': int(L * H),
    }
    save_rows_csv(rsa_per_head_df.to_dict('records'), os.path.join(PHASE_20_DIR, 'rsa_per_head.csv'),
                  'rsa_per_head.csv',
                  columns=['layer', 'head', 'rsa_crel', 'rsa_attr', 'rsa_max', 'rsa_max_before_clamp', 'rsa_crel_pval', 'rsa_attr_pval'])
    save_rows_csv(rsa_partial_df.to_dict('records'), os.path.join(PHASE_20_DIR, 'rsa_per_head_partial.csv'),
                  'rsa_per_head_partial.csv',
                  columns=['layer', 'head', 'rsa_crel_partial', 'rsa_attr_partial', 'rsa_max_partial'])
    save_rows_csv(rsa_sensitivity_no_less_df.to_dict('records'), os.path.join(PHASE_20_DIR, 'rsa_sensitivity_no_less.csv'),
                  'rsa_sensitivity_no_less.csv',
                  columns=['layer', 'head', 'rsa_crel', 'rsa_attr', 'rsa_max', 'rsa_max_before_clamp', 'rsa_crel_pval', 'rsa_attr_pval'])
    save_rows_csv(perturbation_per_head_df.to_dict('records'), os.path.join(PHASE_20_DIR, 'perturbation_per_head.csv'),
                  'perturbation_per_head.csv',
                  columns=['layer', 'head', 'perturbation_L2'])
    save_rows_csv(cell_classification_df.to_dict('records'), os.path.join(PHASE_20_DIR, 'cell_classification.csv'),
                  'cell_classification.csv',
                  columns=['layer', 'head', 'cell', 'rsa_max', 'perturbation_L2', 'importance_quartile'])
    save_json(median_thresholds, os.path.join(PHASE_20_DIR, 'median_thresholds.json'), 'median_thresholds.json')
    phaseB_elapsed_sec = time.time() - phase_b_t0
    save_phase(PHASE_20_DIR, '20_scoring', {
        **expected_cfg,
        'phaseB_elapsed_sec': round(phaseB_elapsed_sec, 3),
        'rsa_elapsed_sec': round(rsa_elapsed_sec, 3),
        'perturbation_elapsed_sec': round(perturbation_elapsed_sec, 3),
        'partial_rsa_elapsed_sec': round(partial_rsa_elapsed_sec, 3),
        'sensitivity_elapsed_sec': round(sensitivity_elapsed_sec, 3),
        'negative_head_count_before_clamp': neg_count,
        'negative_value_range_min': neg_min,
        'negative_value_range_max': neg_max,
        'perturbation_tuples_per_head': 120,
        'cell_counts': cell_counts,
    })
    globals()['rsa_per_head_df'] = rsa_per_head_df
    globals()['perturbation_per_head_df'] = perturbation_per_head_df
    globals()['cell_classification_df'] = cell_classification_df
    log(f'Phase B complete in {phaseB_elapsed_sec:.1f}s; cells={cell_counts}')

_log_mem('Phase B end')


In [ ]:
# ============================================================
# Cell 9: Phase C — Individual Patching, Group Dose-Response, γ Flip Rates
# Purpose:
#   Implements Novel-Word Phase C for unified_pipeline_v7_5_0.ipynb.
#   C.1: per-head input-overwrite patching -> 30_patching/individual_head_effects.csv
#   C.2: group patching dose-response -> 30_patching/dose_response_v2.csv
#   γ:   behavioural argmax flips derived from C.2 -> 38_behavioral/flip_rates.csv
#
# Inputs expected from prior cells:
#   L, H, D, arch, model (optional if already loaded), tokenizer (optional),
#   main_acts, main_rows, _td_df, _ad_df,
#   cell_classification_df, rsa_per_head_df, perturbation_per_head_df,
#   log(), ensure_dir(), save_json(), save_rows_csv(), save_phase(),
#   phase_done(), load_model(), unload_model(),
#   PIPELINE_VERSION, SEED, MODEL_ID, MODEL_SHORT, MODEL_DTYPE,
#   LOCAL_BASE, DRIVE_BASE, PHASE_10_DIR, PHASE_11_DIR
#
# Appendix refs:
#   A.4.2 perturbation / source rule alpha
#   A.5   patching operations
#   A.6   unified first-subtoken readout
#
# Memo refs:
# Locked decisions:
#   - input-overwrite patching (Pattern A) only
#   - final token only
#   - sequential local execution (no ThreadPoolExecutor)
#   - signed deltas; trial-level NaN on exception, never zero-fill
# ============================================================

import csv
import gc
import hashlib
import json
import math
import os
import resource
import shutil
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
PHASE_30_DIR = ensure_dir(os.path.join(LOCAL_BASE, '30_patching'))
PHASE_38_DIR = ensure_dir(os.path.join(LOCAL_BASE, '38_behavioral'))
PHASE_30_DRIVE = os.path.join(DRIVE_BASE, '30_patching')
PHASE_38_DRIVE = os.path.join(DRIVE_BASE, '38_behavioral')

CREL_PAIRS = [
    ('SAME', 'OPP'),
    ('SAME', 'MORE'),
    ('SAME', 'LESS'),
    ('OPP', 'MORE'),
    ('OPP', 'LESS'),
    ('MORE', 'LESS'),
]
RATIOS_DOSE_BASE = [0.01, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50]
RATIOS_BEHAVIORAL = [0.05, 0.10, 0.20, 0.30, 0.50]
GROUP_NAMES = [
    'hihp_imp_desc',
    'hilp_imp_desc',
    'C_imp_asc',
    'D_imp_asc',
    'std_imp_asc',
    'D_pert_asc',
    'diag_rank_asc',
]
PHASE_30_REQUIRED = [
    'individual_head_effects.csv',
    'dose_response_v2.csv',
    'config.json',
]
PHASE_38_REQUIRED = [
    'flip_rates.csv',
    'config.json',
]
C1_FIELDNAMES = [
    'layer',
    'head',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'delta_target_logit_signed',
    'delta_source_logit_signed',
]
C2_FIELDNAMES = [
    'model',
    'group',
    'ratio',
    'k',
    'k_actual',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'delta_target_logit_signed',
    'delta_source_logit_signed',
    'argmax_token_id_clean',
    'argmax_token_id_patched',
]
GAMMA_FIELDNAMES = [
    'group',
    'ratio',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'is_argmax_flip',
]
C1_DRIVE_SYNC_EVERY_N_HEADS = 10


def _torch_dtype(dtype_name):
    if dtype_name == 'float32':
        return torch.float32
    if dtype_name == 'float16':
        return torch.float16
    return torch.bfloat16


def _derive_arch_from_model(model_obj):
    cfg = model_obj.config
    text_cfg = getattr(cfg, 'text_config', cfg)
    num_layers = getattr(text_cfg, 'num_hidden_layers', getattr(text_cfg, 'n_layer', None))
    num_heads = getattr(text_cfg, 'num_attention_heads', getattr(text_cfg, 'n_head', None))
    hidden = getattr(text_cfg, 'hidden_size', getattr(text_cfg, 'n_embd', None))
    head_dim = getattr(text_cfg, 'head_dim', None) or (hidden // num_heads if hidden and num_heads else None)
    return {
        'model_id': MODEL_ID,
        'model_short': MODEL_SHORT,
        'num_layers': num_layers,
        'num_heads': num_heads,
        'head_dim': head_dim,
        'hidden_size': hidden,
        'vocab_size': getattr(text_cfg, 'vocab_size', None),
        'total_heads': (num_layers * num_heads) if num_layers and num_heads else None,
        'torch_dtype': MODEL_DTYPE,
    }


def _resolve_saved(x):
    return getattr(x, 'value', x)


def _to_scalar(x):
    y = _resolve_saved(x)
    if isinstance(y, torch.Tensor):
        if y.numel() != 1:
            raise ValueError(f'Expected scalar tensor, got shape={tuple(y.shape)}')
        return y.detach().float().cpu().item()
    return float(y)


def _to_int_scalar(x):
    return int(round(_to_scalar(x)))


def _log_mem(tag):
    rss_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
        log(f'[{tag}] RAM(maxrss)={rss_gb:.2f}GB VRAM alloc={alloc:.2f}GB peak={peak:.2f}GB')
    else:
        log(f'[{tag}] RAM(maxrss)={rss_gb:.2f}GB CUDA=off')


def _copy_dir(src_dir, dst_dir):
    if not os.path.isdir(src_dir):
        return
    ensure_dir(dst_dir)
    for fname in os.listdir(src_dir):
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))


def _pair_sort_key(x):
    try:
        return (0, int(x))
    except Exception:
        s = str(x)
        if s.startswith('pair') and s[4:].isdigit():
            return (0, int(s[4:]))
        return (1, s)


def _norm_pair_id(x):
    try:
        return int(x)
    except Exception:
        s = str(x)
        if s.startswith('pair') and s[4:].isdigit():
            return int(s[4:])
        return s


def _append_rows_csv(path, rows, fieldnames):
    if not rows:
        return
    file_exists = os.path.exists(path) and os.path.getsize(path) > 0
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)


def _read_csv_rows(path, numeric_float_cols=None, numeric_int_cols=None):
    numeric_float_cols = numeric_float_cols or []
    numeric_int_cols = numeric_int_cols or []
    rows = []
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            for col in numeric_int_cols:
                if row.get(col) not in (None, ''):
                    row[col] = int(float(row[col]))
            for col in numeric_float_cols:
                if row.get(col) not in (None, ''):
                    row[col] = float(row[col])
            rows.append(row)
    return rows


def _write_rows_csv(path, rows, fieldnames):
    with open(path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def _remove_file_if_exists(path):
    if os.path.exists(path):
        os.remove(path)


def _sync_resume_artifacts_from_drive(filenames):
    if not os.path.isdir(PHASE_30_DRIVE):
        return
    for fname in filenames:
        src = os.path.join(PHASE_30_DRIVE, fname)
        dst = os.path.join(PHASE_30_DIR, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
            log(f'[Phase C] restored resume artefact from Drive: {fname}')


def _remove_resume_artifacts(filenames):
    for fname in filenames:
        _remove_file_if_exists(os.path.join(PHASE_30_DIR, fname))
        _remove_file_if_exists(os.path.join(PHASE_30_DRIVE, fname))


def _build_resume_manifest(patch_trials):
    trial_fingerprint_payload = json.dumps(
        sorted([
            (
                str(pt['tgt_tid']),
                str(pt['src_tid']),
                str(pt['crel_pair']),
                str(pt['pair_id']),
                str(pt['attr_dim']),
                int(pt['tgt_tok']),
                int(pt['src_tok']),
            )
            for pt in patch_trials
        ]),
        ensure_ascii=True,
        separators=(',', ':'),
    )
    return {
        'pipeline_version': PIPELINE_VERSION,
        'model_id': MODEL_ID,
        'word_seed': globals().get('WORD_SEED', globals().get('SEED', None)),
        'n_pairs': int(globals().get('N_PAIRS', len({_norm_pair_id(pt['pair_id']) for pt in patch_trials}))),
        'n_patch_trials': len(patch_trials),
        'patch_mechanism': 'input_overwrite_v5_1_pattern',
        'patch_mechanism_version': 'v2_canonical_patch_mechanism',
        'source_rule': 'alpha_same_pair_same_attr_cross_crel',
        'trials_fingerprint_sha256': hashlib.sha256(trial_fingerprint_payload.encode('utf-8')).hexdigest(),
    }


def _manifest_mismatches(saved_manifest, current_manifest):
    return [
        key for key, value in current_manifest.items()
        if saved_manifest.get(key) != value
    ]


def _normalize_completed_heads(items):
    out = set()
    for item in items or []:
        out.add((int(item[0]), int(item[1])))
    return out


def _normalize_completed_combos(items):
    out = set()
    for item in items or []:
        out.add((str(item[0]), round(float(item[1]), 6)))
    return out


def _sorted_completed_heads(completed_heads):
    return [[int(layer_idx), int(head_idx)] for (layer_idx, head_idx) in sorted(completed_heads)]


def _sorted_completed_combos(completed_combos):
    return [[group_name, float(ratio)] for (group_name, ratio) in sorted(completed_combos, key=lambda x: (x[0], x[1]))]


def _save_c1_checkpoint(path, completed_heads, elapsed_sec, manifest, n_tuples_per_head):
    save_json({
        'heads_completed': _sorted_completed_heads(completed_heads),
        'elapsed_sec': round(float(elapsed_sec), 6),
        'n_rows_partial': int(len(completed_heads) * n_tuples_per_head),
        'n_tuples_per_head': int(n_tuples_per_head),
        'manifest': manifest,
    }, path, 'checkpoint_c1.json')


def _save_c2_checkpoint(path, completed_combos, elapsed_sec, manifest, n_tuples_per_combo):
    save_json({
        'completed_combos': _sorted_completed_combos(completed_combos),
        'elapsed_sec': round(float(elapsed_sec), 6),
        'n_rows_partial': int(len(completed_combos) * n_tuples_per_combo),
        'n_tuples_per_combo': int(n_tuples_per_combo),
        'manifest': manifest,
    }, path, 'checkpoint_c2.json')


def _load_c1_resume_state(partial_path, checkpoint_path, patch_trials, manifest):
    filenames = ['individual_head_effects_partial.csv', 'checkpoint_c1.json']
    _sync_resume_artifacts_from_drive(filenames)
    has_partial = os.path.exists(partial_path)
    has_checkpoint = os.path.exists(checkpoint_path)
    if not has_partial and not has_checkpoint:
        return [], set(), 0.0
    if has_partial != has_checkpoint:
        log('[Phase C.1] WARNING incomplete resume state detected -> discarding partial + checkpoint and starting fresh')
        _remove_resume_artifacts(filenames)
        return [], set(), 0.0

    try:
        with open(checkpoint_path) as f:
            ckpt = json.load(f)
        saved_manifest = ckpt.get('manifest', {})
        mismatches = _manifest_mismatches(saved_manifest, manifest)
        if mismatches:
            log(f'[Phase C.1] WARNING resume manifest mismatch on {mismatches} -> discarding partial + checkpoint and starting fresh')
            log(f'[Phase C.1] saved manifest: {saved_manifest}')
            log(f'[Phase C.1] current manifest: {manifest}')
            _remove_resume_artifacts(filenames)
            return [], set(), 0.0

        partial_rows = _read_csv_rows(
            partial_path,
            numeric_float_cols=['delta_target_logit_signed', 'delta_source_logit_signed'],
            numeric_int_cols=['layer', 'head'],
        )
        partial_counts = defaultdict(int)
        for row in partial_rows:
            partial_counts[(int(row['layer']), int(row['head']))] += 1
        complete_heads = {head for head, n_rows in partial_counts.items() if n_rows == len(patch_trials)}
        filtered_rows = [
            row for row in partial_rows
            if (int(row['layer']), int(row['head'])) in complete_heads
        ]
        checkpoint_heads = _normalize_completed_heads(ckpt.get('heads_completed', []))
        row_count_matches = len(filtered_rows) == len(complete_heads) * len(patch_trials)
        agrees = checkpoint_heads == complete_heads and row_count_matches

        if len(filtered_rows) != len(partial_rows):
            dropped_rows = len(partial_rows) - len(filtered_rows)
            log(f'[Phase C.1] WARNING dropping {dropped_rows} partial rows from incomplete heads before resume')
            _write_rows_csv(partial_path, filtered_rows, C1_FIELDNAMES)

        elapsed_prior = float(ckpt.get('elapsed_sec', 0.0))
        if not agrees:
            log(
                '[Phase C.1] WARNING checkpoint/partial mismatch -> using partial as source of truth '
                f'(partial_heads={len(complete_heads)}, checkpoint_heads={len(checkpoint_heads)}, partial_rows={len(filtered_rows)})'
            )
            _save_c1_checkpoint(checkpoint_path, complete_heads, elapsed_prior, manifest, len(patch_trials))

        log(f'[Phase C.1] resume validated: partial rows={len(filtered_rows)} heads complete={len(complete_heads)}')
        return filtered_rows, complete_heads, elapsed_prior
    except Exception as exc:
        log(f'[Phase C.1] WARNING resume load failed ({type(exc).__name__}: {exc}) -> discarding partial + checkpoint and starting fresh')
        _remove_resume_artifacts(filenames)
        return [], set(), 0.0


def _load_c2_resume_state(partial_path, checkpoint_path, patch_trials, manifest):
    filenames = ['dose_response_v2_partial.csv', 'checkpoint_c2.json']
    _sync_resume_artifacts_from_drive(filenames)
    has_partial = os.path.exists(partial_path)
    has_checkpoint = os.path.exists(checkpoint_path)
    if not has_partial and not has_checkpoint:
        return [], set(), 0.0
    if has_partial != has_checkpoint:
        log('[Phase C.2] WARNING incomplete resume state detected -> discarding partial + checkpoint and starting fresh')
        _remove_resume_artifacts(filenames)
        return [], set(), 0.0

    try:
        with open(checkpoint_path) as f:
            ckpt = json.load(f)
        saved_manifest = ckpt.get('manifest', {})
        mismatches = _manifest_mismatches(saved_manifest, manifest)
        if mismatches:
            log(f'[Phase C.2] WARNING resume manifest mismatch on {mismatches} -> discarding partial + checkpoint and starting fresh')
            log(f'[Phase C.2] saved manifest: {saved_manifest}')
            log(f'[Phase C.2] current manifest: {manifest}')
            _remove_resume_artifacts(filenames)
            return [], set(), 0.0

        partial_rows = _read_csv_rows(
            partial_path,
            numeric_float_cols=[
                'ratio',
                'delta_target_logit_signed',
                'delta_source_logit_signed',
                'argmax_token_id_clean',
                'argmax_token_id_patched',
            ],
            numeric_int_cols=['k', 'k_actual'],
        )
        partial_counts = defaultdict(int)
        for row in partial_rows:
            combo_key = (str(row['group']), round(float(row['ratio']), 6))
            partial_counts[combo_key] += 1
        complete_combos = {combo for combo, n_rows in partial_counts.items() if n_rows == len(patch_trials)}
        filtered_rows = [
            row for row in partial_rows
            if (str(row['group']), round(float(row['ratio']), 6)) in complete_combos
        ]
        checkpoint_combos = _normalize_completed_combos(ckpt.get('completed_combos', []))
        row_count_matches = len(filtered_rows) == len(complete_combos) * len(patch_trials)
        agrees = checkpoint_combos == complete_combos and row_count_matches

        if len(filtered_rows) != len(partial_rows):
            dropped_rows = len(partial_rows) - len(filtered_rows)
            log(f'[Phase C.2] WARNING dropping {dropped_rows} partial rows from incomplete combos before resume')
            _write_rows_csv(partial_path, filtered_rows, C2_FIELDNAMES)

        elapsed_prior = float(ckpt.get('elapsed_sec', 0.0))
        if not agrees:
            log(
                '[Phase C.2] WARNING checkpoint/partial mismatch -> using partial as source of truth '
                f'(partial_combos={len(complete_combos)}, checkpoint_combos={len(checkpoint_combos)}, partial_rows={len(filtered_rows)})'
            )
            _save_c2_checkpoint(checkpoint_path, complete_combos, elapsed_prior, manifest, len(patch_trials))

        log(f'[Phase C.2] resume validated: partial rows={len(filtered_rows)} combos complete={len(complete_combos)}')
        return filtered_rows, complete_combos, elapsed_prior
    except Exception as exc:
        log(f'[Phase C.2] WARNING resume load failed ({type(exc).__name__}: {exc}) -> discarding partial + checkpoint and starting fresh')
        _remove_resume_artifacts(filenames)
        return [], set(), 0.0


def _finalize_resume_outputs(partial_path, final_path, checkpoint_path, cleanup_filenames):
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(partial_path, final_path)
    _remove_file_if_exists(checkpoint_path)
    _remove_resume_artifacts(cleanup_filenames)


def _load_main_logits_dict():
    if 'main_logits' in globals() and isinstance(main_logits, dict):
        return main_logits
    local_npz = os.path.join(PHASE_10_DIR, 'clean_logits.npz')
    drive_npz = os.path.join(DRIVE_BASE, '10_collection', 'clean_logits.npz')
    if not os.path.exists(local_npz) and os.path.exists(drive_npz):
        shutil.copy2(drive_npz, local_npz)
    if not os.path.exists(local_npz):
        log('WARNING clean_logits.npz missing; clean metrics will fall back to trace forwards')
        return None
    npz = np.load(local_npz)
    out = {k: npz[k] for k in npz.files}
    npz.close()
    log(f'[Phase C] loaded clean logits from disk: {len(out)} trials')
    return out


def _build_trial_tables(trials_df, diag_df, tokenizer_obj):
    if 'prompt' not in trials_df.columns:
        raise KeyError('trial_definitions missing prompt column')
    trial_rows = trials_df.to_dict('records')
    trial_by_key = {}
    prompt_by_tid = {}
    diag_lookup = {}
    if diag_df is not None and len(diag_df):
        diag_lookup = {r['trial_id']: r for r in diag_df.to_dict('records')}
    for row in trial_rows:
        key = (row['crel'], _norm_pair_id(row['pair_id']), row['attr_dim'])
        trial_by_key[key] = row
        prompt_by_tid[row['trial_id']] = row['prompt']

    def _first_subtok_from_row(row):
        diag_row = diag_lookup.get(row['trial_id'])
        if diag_row is not None and 'correct_first_subtoken_id' in diag_row and pd.notna(diag_row['correct_first_subtoken_id']):
            return int(diag_row['correct_first_subtoken_id'])
        ids = tokenizer_obj.encode(' ' + str(row['correct_answer']), add_special_tokens=False)
        if not ids:
            raise ValueError(f'Empty encoding for correct_answer={row["correct_answer"]!r}')
        return int(ids[0])

    patch_trials = []
    pair_ids_sorted = sorted(trials_df['pair_id'].unique().tolist(), key=_pair_sort_key)
    attr_dims_sorted = sorted(trials_df['attr_dim'].unique().tolist())
    for c1, c2 in CREL_PAIRS:
        for pair_id_raw in pair_ids_sorted:
            pair_id_norm = _norm_pair_id(pair_id_raw)
            for attr_dim in attr_dims_sorted:
                tgt_row = trial_by_key.get((c1, pair_id_norm, attr_dim))
                src_row = trial_by_key.get((c2, pair_id_norm, attr_dim))
                if tgt_row is None or src_row is None:
                    raise KeyError(f'Missing alpha-rule trial for {(c1, c2, pair_id_raw, attr_dim)}')
                patch_trials.append({
                    'crel_pair': f'{c1}-{c2}',
                    'pair_id': tgt_row['pair_id'],
                    'attr_dim': attr_dim,
                    'tgt_tid': tgt_row['trial_id'],
                    'src_tid': src_row['trial_id'],
                    'tgt_tok': _first_subtok_from_row(tgt_row),
                    'src_tok': _first_subtok_from_row(src_row),
                })
    if len(patch_trials) != 120:
        raise AssertionError(f'Expected 120 alpha tuples, got {len(patch_trials)}')
    return patch_trials, prompt_by_tid


def _build_group_orderings(cell_df, rsa_df, pert_df):
    merged = (
        cell_df[['layer', 'head', 'cell']]
        .merge(rsa_df[['layer', 'head', 'rsa_max']], on=['layer', 'head'], how='left')
        .merge(pert_df[['layer', 'head', 'perturbation_L2']], on=['layer', 'head'], how='left')
        .copy()
    )
    merged['layer'] = merged['layer'].astype(int)
    merged['head'] = merged['head'].astype(int)
    merged['rsa_max'] = merged['rsa_max'].astype(float)
    merged['perturbation_L2'] = merged['perturbation_L2'].astype(float)
    merged['rsa_rank_asc'] = merged['rsa_max'].rank(method='average', ascending=True)
    merged['pert_rank_asc'] = merged['perturbation_L2'].rank(method='average', ascending=True)
    merged['diag_rank_sum'] = merged['rsa_rank_asc'] + merged['pert_rank_asc']

    def _heads(df_slice, cols, ascending):
        ordered = df_slice.sort_values(cols, ascending=ascending, kind='mergesort')
        return list(zip(ordered['layer'].astype(int), ordered['head'].astype(int)))

    group_defs = {
        'hihp_imp_desc': _heads(
            merged[merged['cell'] == 'hihp'],
            ['rsa_max', 'layer', 'head'],
            [False, True, True],
        ),
        'hilp_imp_desc': _heads(
            merged[merged['cell'] == 'hilp'],
            ['rsa_max', 'layer', 'head'],
            [False, True, True],
        ),
        'C_imp_asc': _heads(
            merged[merged['cell'] == 'C'],
            ['rsa_max', 'layer', 'head'],
            [True, True, True],
        ),
        'D_imp_asc': _heads(
            merged[merged['cell'] == 'D'],
            ['rsa_max', 'layer', 'head'],
            [True, True, True],
        ),
        'std_imp_asc': _heads(
            merged,
            ['rsa_max', 'layer', 'head'],
            [True, True, True],
        ),
        'D_pert_asc': _heads(
            merged[merged['cell'] == 'D'],
            ['perturbation_L2', 'layer', 'head'],
            [True, True, True],
        ),
        'diag_rank_asc': _heads(
            merged,
            ['diag_rank_sum', 'rsa_max', 'perturbation_L2', 'layer', 'head'],
            [True, True, True, True, True],
        ),
    }
    cell_counts = {
        cell: int((merged['cell'] == cell).sum())
        for cell in ['hihp', 'hilp', 'C', 'D']
    }
    return merged, group_defs, cell_counts


def _prepare_ratio_grid(base_ratios, total_heads, max_equal_k_ratio):
    adjustments = []
    if total_heads <= 0:
        raise ValueError('total_heads must be positive')
    max_half_ratio = (total_heads // 2) / float(total_heads)
    raw_ratios = sorted(set(list(base_ratios) + [max_equal_k_ratio]))
    ratio_by_k = {}
    for raw_ratio in raw_ratios:
        capped_ratio = min(float(raw_ratio), max_half_ratio)
        if capped_ratio != raw_ratio:
            adjustments.append(f'ratio {raw_ratio:.6f} capped to {capped_ratio:.6f} by floor(total_heads/2)')
        k = int(capped_ratio * total_heads)
        if k <= 0:
            k = 1
            adjustments.append(f'ratio {capped_ratio:.6f} mapped to k=0 -> raised to k=1')
        kept = ratio_by_k.get(k)
        if kept is None:
            ratio_by_k[k] = capped_ratio
        elif capped_ratio < kept:
            adjustments.append(f'dedup k={k}: kept smaller ratio {capped_ratio:.6f}, dropped {kept:.6f}')
            ratio_by_k[k] = capped_ratio
        else:
            adjustments.append(f'dedup k={k}: kept smaller ratio {kept:.6f}, dropped {capped_ratio:.6f}')
    ratio_grid = [round(v, 6) for _, v in sorted(ratio_by_k.items(), key=lambda kv: kv[0])]
    applied_ratio = round(min(float(max_equal_k_ratio), max_half_ratio), 6)
    applied_k = max(1, int(applied_ratio * total_heads))
    if applied_k not in ratio_by_k:
        ratio_grid.append(applied_ratio)
        ratio_grid = sorted(set(ratio_grid))
        adjustments.append(f'forced max_equal_k_ratio={applied_ratio:.6f} into grid')
    else:
        kept = round(ratio_by_k[applied_k], 6)
        if kept != applied_ratio:
            adjustments.append(
                f'max_equal_k_ratio={applied_ratio:.6f} maps to existing k={applied_k}; '
                f'effective kept ratio={kept:.6f}'
            )
    k_by_ratio = {round(v, 6): int(k) for k, v in ratio_by_k.items()}
    # Collision check by rounded RATIO key, not k value. Adding applied_ratio only
    # when its rounded form is not already keyed by a full-precision ratio (which
    # would have the source-of-truth k from int(full_ratio * N), not int(rounded * N)).
    if applied_ratio not in k_by_ratio:
        k_by_ratio[applied_ratio] = max(1, int(applied_k))
    if adjustments:
        for msg in adjustments:
            log(f'[Phase C ratios] {msg}')
    log(f'[Phase C ratios] final grid={ratio_grid}')
    log(f'[Phase C ratios] k_by_ratio={k_by_ratio}')
    return ratio_grid, applied_ratio, k_by_ratio


def _clean_metrics_from_trace(model_obj, prompt, target_token, source_token):
    with model_obj.trace(prompt):
        tgt_proxy = model_obj.output.logits[0, -1, target_token].save()
        src_proxy = model_obj.output.logits[0, -1, source_token].save()
        argmax_proxy = torch.argmax(model_obj.output.logits[0, -1, :], dim=-1).save()
    return {
        'clean_target_logit': _to_scalar(tgt_proxy),
        'clean_source_logit': _to_scalar(src_proxy),
        'argmax_clean': _to_int_scalar(argmax_proxy),
    }


def _cache_clean_metrics(patch_trials, prompt_by_tid, clean_logits_dict, model_obj):
    cache = {}
    unique_keys = {}
    for pt in patch_trials:
        key = (pt['tgt_tid'], int(pt['tgt_tok']), int(pt['src_tok']))
        unique_keys[key] = pt
    log(f'[Phase C] caching clean metrics for {len(unique_keys)} unique (trial,target,source) tuples')
    for i, (key, pt) in enumerate(unique_keys.items(), start=1):
        tgt_tid, tgt_tok, src_tok = key
        try:
            if clean_logits_dict is not None and tgt_tid in clean_logits_dict:
                logits = clean_logits_dict[tgt_tid]
                cache[key] = {
                    'clean_target_logit': float(logits[tgt_tok]),
                    'clean_source_logit': float(logits[src_tok]),
                    'argmax_clean': int(np.argmax(logits)),
                }
            else:
                cache[key] = _clean_metrics_from_trace(model_obj, prompt_by_tid[tgt_tid], tgt_tok, src_tok)
        except Exception as exc:
            log(f'WARNING clean metric cache failed for {key}: {type(exc).__name__}: {exc}')
            cache[key] = {
                'clean_target_logit': float('nan'),
                'clean_source_logit': float('nan'),
                'argmax_clean': np.nan,
            }
        if i % 20 == 0 or i == len(unique_keys):
            log(f'[Phase C] clean metric cache {i}/{len(unique_keys)}')
    return cache


def _apply_output_delta(layer_proxy, head_idx, patch_head_act, head_dim, hidden_size, model_dtype):
    """Pattern A (v5.1.1 proven): input-slice overwrite.
    Envoy-safe: each head writes a disjoint slice of proj.input; no repeated reads.
    Replaces the former output-delta path that mutated `o_proj` outputs, which
    triggered nnsight OutOfOrderError on multi-head group patching (2026-04-21)."""
    proj = layer_proxy.self_attn.o_proj
    col_s = int(head_idx) * int(head_dim)
    # Cast source to native dtype for input slot compatibility (forward runs in bf16)
    source = patch_head_act.to(dtype=model_dtype)
    proj.input[0, -1, col_s:col_s + int(head_dim)] = source
    # NO proj.output manipulation — forward pass will re-compute with patched input


def _trace_single_head_patch(model_obj, prompt, layer_idx, head_idx, patch_head_act, target_token, source_token, head_dim, hidden_size, model_dtype):
    with model_obj.trace(prompt):
        _apply_output_delta(
            model_obj.model.layers[layer_idx],
            head_idx,
            patch_head_act,
            head_dim,
            hidden_size,
            model_dtype,
        )
        tgt_proxy = model_obj.output.logits[0, -1, target_token].save()
        src_proxy = model_obj.output.logits[0, -1, source_token].save()
    return _to_scalar(tgt_proxy), _to_scalar(src_proxy)


def _trace_batched_head_patch(
    model_obj,
    prompt,
    source_tensor_all_heads,
    head_batch,
    target_token,
    source_token,
    head_dim,
    model_dtype,
):
    """Pattern A batched over heads: patch B different single heads in B batch elements.
    v5.1.1/v4.1 proven speedup pattern, reintroduced to v7.5.0 for large models."""
    B = len(head_batch)
    if not isinstance(source_tensor_all_heads, torch.Tensor):
        src_t = torch.as_tensor(source_tensor_all_heads)
    else:
        src_t = source_tensor_all_heads
    src_t = src_t.to(device='cuda', dtype=model_dtype)
    prompts_batch = [prompt] * B
    with model_obj.trace(prompts_batch):
        for bi, (l_idx, h_idx) in enumerate(head_batch):
            proj = model_obj.model.layers[int(l_idx)].self_attn.o_proj
            col_s = int(h_idx) * int(head_dim)
            proj.input[bi, -1, col_s:col_s + int(head_dim)] = src_t[int(l_idx), int(h_idx)]
        tgt_proxy = model_obj.output.logits[:, -1, int(target_token)].save()
        src_proxy = model_obj.output.logits[:, -1, int(source_token)].save()
    tgt_arr = _resolve_saved(tgt_proxy).detach().float().cpu().numpy()
    src_arr = _resolve_saved(src_proxy).detach().float().cpu().numpy()
    return tgt_arr, src_arr


def _trace_group_patch(model_obj, prompt, source_tensor, group_heads, target_token, source_token, head_dim, hidden_size, model_dtype):
    sorted_heads = sorted(group_heads, key=lambda lh: (int(lh[0]), int(lh[1])))
    with model_obj.trace(prompt):
        for layer_idx, head_idx in sorted_heads:
            _apply_output_delta(
                model_obj.model.layers[layer_idx],
                head_idx,
                source_tensor[layer_idx, head_idx],
                head_dim,
                hidden_size,
                model_dtype,
            )
        tgt_proxy = model_obj.output.logits[0, -1, target_token].save()
        src_proxy = model_obj.output.logits[0, -1, source_token].save()
        argmax_proxy = torch.argmax(model_obj.output.logits[0, -1, :], dim=-1).save()
    return _to_scalar(tgt_proxy), _to_scalar(src_proxy), _to_int_scalar(argmax_proxy)


def _run_c1(
    model_obj,
    patch_trials,
    prompt_by_tid,
    source_tensor_cache,
    clean_metric_cache,
    total_heads,
    head_dim,
    hidden_size,
    model_dtype,
    resume_manifest,
):
    """Phase C.1 — individual-head patching with HEAD_BATCH_SIZE prompt-replication speedup.
    Pattern A input-slice overwrite: B heads patched in B batch elements in a single forward pass.
    Speedup ~30-60x vs v7.5.0 batch=1. OOM ladder fallback per head_batch.
    """
    global HEAD_BATCH_SIZE
    partial_path = os.path.join(PHASE_30_DIR, 'individual_head_effects_partial.csv')
    checkpoint_path = os.path.join(PHASE_30_DIR, 'checkpoint_c1.json')
    final_path = os.path.join(PHASE_30_DIR, 'individual_head_effects.csv')

    partial_rows, completed_heads, elapsed_prior = _load_c1_resume_state(
        partial_path=partial_path,
        checkpoint_path=checkpoint_path,
        patch_trials=patch_trials,
        manifest=resume_manifest,
    )

    all_heads = [(l_idx, h_idx) for l_idx in range(L) for h_idx in range(H)]
    remaining_heads = [lh for lh in all_heads if lh not in completed_heads]
    t0 = time.time()
    processed_this_run = 0
    ladder = list(HEAD_BATCH_SIZE_LADDER)
    if HEAD_BATCH_SIZE not in ladder:
        ladder = [HEAD_BATCH_SIZE] + [b for b in ladder if b < HEAD_BATCH_SIZE]
    hbs_ptr = 0
    current_hbs = ladder[hbs_ptr]
    log(f'[Phase C.1] start: {len(remaining_heads)}/{total_heads} heads remaining, HEAD_BATCH_SIZE={current_hbs}, ladder={ladder}')

    def _process_head_batch(head_batch):
        head_trial_rows = {lh: [] for lh in head_batch}
        for pt in patch_trials:
            cache_key = (pt['tgt_tid'], int(pt['tgt_tok']), int(pt['src_tok']))
            clean = clean_metric_cache[cache_key]
            prompt = prompt_by_tid[pt['tgt_tid']]
            source_all = source_tensor_cache[pt['src_tid']]
            tgt_arr, src_arr = _trace_batched_head_patch(
                model_obj=model_obj,
                prompt=prompt,
                source_tensor_all_heads=source_all,
                head_batch=head_batch,
                target_token=int(pt['tgt_tok']),
                source_token=int(pt['src_tok']),
                head_dim=head_dim,
                model_dtype=model_dtype,
            )
            for bi, (l_idx, h_idx) in enumerate(head_batch):
                dt = float(tgt_arr[bi]) - clean['clean_target_logit']
                ds = float(src_arr[bi]) - clean['clean_source_logit']
                head_trial_rows[(l_idx, h_idx)].append({
                    'layer': int(l_idx),
                    'head': int(h_idx),
                    'crel_pair': pt['crel_pair'],
                    'pair_id': pt['pair_id'],
                    'attr_dim': pt['attr_dim'],
                    'delta_target_logit_signed': dt,
                    'delta_source_logit_signed': ds,
                })
        return head_trial_rows

    i = 0
    while i < len(remaining_heads):
        head_batch = remaining_heads[i:i + current_hbs]
        try:
            head_trial_rows = _process_head_batch(head_batch)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as oom:
            msg = str(oom)
            is_oom = isinstance(oom, torch.cuda.OutOfMemoryError) or ('out of memory' in msg.lower())
            if not is_oom or hbs_ptr >= len(ladder) - 1:
                log(f'[Phase C.1] non-OOM error at i={i}: {type(oom).__name__}: {msg[:160]}; fallback to single-head')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                head_trial_rows = {}
                for lh in head_batch:
                    try:
                        r = _process_head_batch([lh])
                        head_trial_rows.update(r)
                    except Exception as exc2:
                        log(f'WARNING C1 single-head failed at layer={lh[0]} head={lh[1]}: {type(exc2).__name__}: {exc2}')
                        head_trial_rows[lh] = [{
                            'layer': int(lh[0]), 'head': int(lh[1]),
                            'crel_pair': pt['crel_pair'], 'pair_id': pt['pair_id'], 'attr_dim': pt['attr_dim'],
                            'delta_target_logit_signed': float('nan'),
                            'delta_source_logit_signed': float('nan'),
                        } for pt in patch_trials]
            else:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
                hbs_ptr += 1
                current_hbs = ladder[hbs_ptr]
                log(f'[Phase C.1] OOM at HEAD_BATCH_SIZE={len(head_batch)} → ladder down to {current_hbs}')
                continue

        for lh in head_batch:
            rows = head_trial_rows[lh]
            _append_rows_csv(partial_path, rows, C1_FIELDNAMES)
            partial_rows.extend(rows)
            completed_heads.add(lh)
            processed_this_run += 1
        elapsed_total = elapsed_prior + (time.time() - t0)
        _save_c1_checkpoint(
            checkpoint_path,
            completed_heads,
            elapsed_total,
            resume_manifest,
            len(patch_trials),
        )
        i += len(head_batch)

        if (len(completed_heads) % C1_DRIVE_SYNC_EVERY_N_HEADS) < current_hbs or len(completed_heads) == total_heads:
            rate = processed_this_run / max(time.time() - t0, 1e-9)
            remaining = total_heads - len(completed_heads)
            eta_hr = remaining / max(rate, 1e-9) / 3600.0
            log(
                f'[Phase C.1] heads {len(completed_heads)}/{total_heads} '
                f'elapsed={elapsed_total:.0f}s ETA={eta_hr:.2f}h HBS={current_hbs}'
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            _log_mem('Phase C.1 progress')
            if globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
                save_to_drive(PHASE_30_DIR, '30_patching')

    _finalize_resume_outputs(
        partial_path=partial_path,
        final_path=final_path,
        checkpoint_path=checkpoint_path,
        cleanup_filenames=['individual_head_effects_partial.csv', 'checkpoint_c1.json'],
    )
    c1_elapsed_sec = elapsed_prior + (time.time() - t0)
    log(f'[Phase C.1] complete: {len(completed_heads)} heads, elapsed={c1_elapsed_sec:.1f}s final HBS={current_hbs}')
    if globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
        save_to_drive(PHASE_30_DIR, '30_patching')
    return c1_elapsed_sec, len(completed_heads)


def _run_c2(
    model_obj,
    patch_trials,
    prompt_by_tid,
    source_tensor_cache,
    clean_metric_cache,
    group_defs,
    ratios_dose,
    total_heads,
    head_dim,
    hidden_size,
    model_dtype,
    resume_manifest,
    k_by_ratio=None,
):
    partial_path = os.path.join(PHASE_30_DIR, 'dose_response_v2_partial.csv')
    checkpoint_path = os.path.join(PHASE_30_DIR, 'checkpoint_c2.json')
    dose_path = os.path.join(PHASE_30_DIR, 'dose_response_v2.csv')
    rows, completed_combos, elapsed_prior = _load_c2_resume_state(
        partial_path=partial_path,
        checkpoint_path=checkpoint_path,
        patch_trials=patch_trials,
        manifest=resume_manifest,
    )

    t0 = time.time()
    combo_total = len(GROUP_NAMES) * len(ratios_dose)
    combo_done = len(completed_combos)
    combo_done_start = combo_done
    for group_name in GROUP_NAMES:
        ordered_heads = group_defs.get(group_name, [])
        for ratio in ratios_dose:
            ratio_key = round(float(ratio), 6)
            combo_key = (group_name, ratio_key)
            if combo_key in completed_combos:
                continue
            if k_by_ratio is not None and ratio_key in k_by_ratio:
                k = max(1, int(k_by_ratio[ratio_key]))
            else:
                k = max(1, int(ratio * total_heads))
            k_actual = min(k, len(ordered_heads))
            active_heads = ordered_heads[:k_actual]
            combo_rows = []
            for pt in patch_trials:
                cache_key = (pt['tgt_tid'], int(pt['tgt_tok']), int(pt['src_tok']))
                clean = clean_metric_cache[cache_key]
                try:
                    if k_actual == 0:
                        raise RuntimeError(f'group {group_name} has zero available heads')
                    patched_tgt, patched_src, argmax_patched = _trace_group_patch(
                        model_obj=model_obj,
                        prompt=prompt_by_tid[pt['tgt_tid']],
                        source_tensor=source_tensor_cache[pt['src_tid']],
                        group_heads=active_heads,
                        target_token=int(pt['tgt_tok']),
                        source_token=int(pt['src_tok']),
                        head_dim=head_dim,
                        hidden_size=hidden_size,
                        model_dtype=model_dtype,
                    )
                    dt = patched_tgt - clean['clean_target_logit']
                    ds = patched_src - clean['clean_source_logit']
                except Exception as exc:
                    log(
                        f'WARNING C2 patch failed group={group_name} ratio={ratio:.6f} '
                        f'trial={pt["tgt_tid"]}: {type(exc).__name__}: {exc}'
                    )
                    dt = float('nan')
                    ds = float('nan')
                    argmax_patched = np.nan
                combo_rows.append({
                    'model': MODEL_SHORT,
                    'group': group_name,
                    'ratio': ratio,
                    'k': k,
                    'k_actual': k_actual,
                    'crel_pair': pt['crel_pair'],
                    'pair_id': pt['pair_id'],
                    'attr_dim': pt['attr_dim'],
                    'delta_target_logit_signed': dt,
                    'delta_source_logit_signed': ds,
                    'argmax_token_id_clean': clean['argmax_clean'],
                    'argmax_token_id_patched': argmax_patched,
                })
            _append_rows_csv(partial_path, combo_rows, C2_FIELDNAMES)
            rows.extend(combo_rows)
            completed_combos.add(combo_key)
            combo_done += 1
            elapsed_total = elapsed_prior + (time.time() - t0)
            _save_c2_checkpoint(
                checkpoint_path,
                completed_combos,
                elapsed_total,
                resume_manifest,
                len(patch_trials),
            )
            processed_this_run = combo_done - combo_done_start
            rate = processed_this_run / max(time.time() - t0, 1e-9) if processed_this_run > 0 else 0.0
            eta_min = (combo_total - combo_done) / max(rate, 1e-9) / 60.0 if combo_done < combo_total and rate > 0 else 0.0
            log(
                f'[Phase C.2] group={group_name} ratio={ratio:.6f} '
                f'k={k} k_actual={k_actual} done {combo_done}/{combo_total} '
                f'elapsed={elapsed_total:.0f}s ETA={eta_min:.1f}m'
            )
            if globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
                save_to_drive(PHASE_30_DIR, '30_patching')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    _finalize_resume_outputs(
        partial_path=partial_path,
        final_path=dose_path,
        checkpoint_path=checkpoint_path,
        cleanup_filenames=['dose_response_v2_partial.csv', 'checkpoint_c2.json'],
    )
    c2_elapsed_sec = elapsed_prior + (time.time() - t0)
    log(f'[Phase C.2] complete: rows={len(rows)} elapsed={c2_elapsed_sec:.1f}s')
    if globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
        save_to_drive(PHASE_30_DIR, '30_patching')
    return pd.DataFrame(rows), c2_elapsed_sec


def _derive_gamma(dose_df):
    kept = dose_df[dose_df['ratio'].round(6).isin({round(r, 6) for r in RATIOS_BEHAVIORAL})].copy()
    gamma_rows = []
    for row in kept.to_dict('records'):
        clean_argmax = row['argmax_token_id_clean']
        patched_argmax = row['argmax_token_id_patched']
        if pd.isna(clean_argmax) or pd.isna(patched_argmax):
            flip = np.nan
        else:
            flip = int(int(clean_argmax) != int(patched_argmax))
        gamma_rows.append({
            'group': row['group'],
            'ratio': row['ratio'],
            'crel_pair': row['crel_pair'],
            'pair_id': row['pair_id'],
            'attr_dim': row['attr_dim'],
            'is_argmax_flip': flip,
        })
    gamma_df = pd.DataFrame(gamma_rows, columns=GAMMA_FIELDNAMES)
    agg = {}
    for group_name, df_g in gamma_df.groupby('group'):
        agg[group_name] = {}
        for ratio, df_gr in df_g.groupby('ratio'):
            vals = pd.to_numeric(df_gr['is_argmax_flip'], errors='coerce')
            agg[group_name][f'{float(ratio):.6f}'] = float(vals.mean()) if vals.notna().any() else None
    return gamma_df, agg


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
_log_mem('Phase C start')

if 'model' not in globals() or model is None:
    log('[Phase C] model not in globals -> loading')
    model, tokenizer, arch = load_model(MODEL_ID, MODEL_DTYPE)
else:
    tokenizer = globals().get('tokenizer', getattr(model, 'tokenizer', None))
    arch = globals().get('arch', None) or _derive_arch_from_model(model)
    log('[Phase C] reusing existing loaded model')

L = globals().get('L', arch['num_layers'])
H = globals().get('H', arch['num_heads'])
D = globals().get('D', arch['head_dim'])
HIDDEN = globals().get('HIDDEN', arch.get('hidden_size') or (H * D))
TOTAL_HEADS = int(L * H)
MODEL_TORCH_DTYPE = _torch_dtype(MODEL_DTYPE)

if tokenizer is None:
    tokenizer = getattr(model, 'tokenizer', None)
if tokenizer is None:
    raise RuntimeError('Tokenizer unavailable after model resolution')

for layer_idx in range(L):
    proj = model.model.layers[layer_idx].self_attn.o_proj
    assert getattr(proj, 'bias', None) is None, f'o_proj.bias must be None at layer {layer_idx}'

patch_trials, prompt_by_tid = _build_trial_tables(_td_df.copy(), _ad_df.copy(), tokenizer)
resume_manifest = _build_resume_manifest(patch_trials)
_, group_defs, cell_counts = _build_group_orderings(
    cell_classification_df.copy(),
    rsa_per_head_df.copy(),
    perturbation_per_head_df.copy(),
)
max_equal_k_ratio = (min(cell_counts.get('C', 0), cell_counts.get('D', 0)) / float(TOTAL_HEADS)) if TOTAL_HEADS > 0 else 0.0
RATIOS_DOSE, MAX_EQUAL_K_RATIO_APPLIED, K_BY_RATIO = _prepare_ratio_grid(
    RATIOS_DOSE_BASE,
    TOTAL_HEADS,
    max_equal_k_ratio,
)

expected_cfg_c = {
    'phase_name': '30_patching',
    'source_rule': 'alpha_same_pair_same_attr_cross_crel',
    'patch_mechanism': 'input_overwrite_v5_1_pattern',
    'patch_mechanism_version': 'v2_canonical_patch_mechanism',
    'ratios_dose': RATIOS_DOSE,
    'groups': GROUP_NAMES,
    'max_equal_k_ratio_applied': MAX_EQUAL_K_RATIO_APPLIED,
    'total_heads': TOTAL_HEADS,
    'n_tuples_per_head': len(patch_trials),
    'manifest': resume_manifest,
}
expected_cfg_gamma = {
    'phase_name': '38_behavioral',
    'ratios_behavioral': RATIOS_BEHAVIORAL,
    'derived_from': 'dose_response_v2.csv',
}

# === Canonical equal-count dose-response construction (idempotent, robust) ===
# Detects off-by-one rows at the equal-k ratio in any of: cached final/partial, local final/partial.
# If detected, surgically split data so _run_c2 re-processes only equal-k combos
# and _run_c1 returns instantly via completed_heads.
# Note: Phase 38 (RATIOS_BEHAVIORAL=[0.05,0.10,0.20,0.30,0.50]) does NOT include
# equal-k ratios for Llama-8B (0.173828) / Gemma-2-9B (0.163690) / Llama-70B
# (0.153906). flip_rates.csv content is invariant.
_eq_ratio_key = round(float(MAX_EQUAL_K_RATIO_APPLIED), 6)
_correct_k_at_eq = int(K_BY_RATIO.get(
    _eq_ratio_key,
    max(1, int(MAX_EQUAL_K_RATIO_APPLIED * TOTAL_HEADS)),
))
_dose_candidates = [
    os.path.join(PHASE_30_DRIVE, 'dose_response_v2.csv'),
    os.path.join(PHASE_30_DRIVE, 'dose_response_v2_partial.csv'),
    os.path.join(PHASE_30_DIR, 'dose_response_v2.csv'),
    os.path.join(PHASE_30_DIR, 'dose_response_v2_partial.csv'),
]
_existing_dose_path = next((_p for _p in _dose_candidates if os.path.exists(_p)), None)
_migration_needed = False
_saved_k_at_eq = None
_existing_dose_df = None
if _existing_dose_path is not None:
    _existing_dose_df = pd.read_csv(_existing_dose_path)
    _eq_mask = _existing_dose_df['ratio'].round(6) == _eq_ratio_key
    if _eq_mask.any():
        _saved_k_at_eq = int(_existing_dose_df.loc[_eq_mask, 'k'].iloc[0])
        if _saved_k_at_eq != _correct_k_at_eq:
            _migration_needed = True

if _migration_needed:
    log(f'[Canonical Phase C setup] equal-count state check: saved k={_saved_k_at_eq} '
        f'!= correct k={_correct_k_at_eq} at ratio={_eq_ratio_key} '
        f'(source={_existing_dose_path}, removing {int(_eq_mask.sum())} off-by-one C2 rows)')
    ensure_dir(PHASE_30_DIR)
    _indiv_candidates = [
        os.path.join(PHASE_30_DRIVE, 'individual_head_effects.csv'),
        os.path.join(PHASE_30_DRIVE, 'individual_head_effects_partial.csv'),
        os.path.join(PHASE_30_DIR, 'individual_head_effects.csv'),
        os.path.join(PHASE_30_DIR, 'individual_head_effects_partial.csv'),
    ]
    _existing_indiv_path = next((_p for _p in _indiv_candidates if os.path.exists(_p)), None)
    _indiv_df_in_mem = pd.read_csv(_existing_indiv_path) if _existing_indiv_path is not None else None
    for _stale in [
        'dose_response_v2_partial.csv', 'checkpoint_c2.json',
        'individual_head_effects_partial.csv', 'checkpoint_c1.json',
    ]:
        for _base in [PHASE_30_DIR, PHASE_30_DRIVE]:
            _p = os.path.join(_base, _stale)
            if os.path.exists(_p):
                try:
                    os.remove(_p)
                except OSError as _e:
                    log(f'[Canonical Phase C setup] WARNING failed to remove cached {_p}: {_e}')
    _kept_dose_df = _existing_dose_df.loc[~_eq_mask].copy()
    _local_c2_partial = os.path.join(PHASE_30_DIR, 'dose_response_v2_partial.csv')
    _local_c2_checkpoint = os.path.join(PHASE_30_DIR, 'checkpoint_c2.json')
    _kept_dose_df.to_csv(_local_c2_partial, index=False)
    _kept_combos = set()
    for _, _row in _kept_dose_df.iterrows():
        _kept_combos.add((str(_row['group']), round(float(_row['ratio']), 6)))
    _save_c2_checkpoint(
        _local_c2_checkpoint, _kept_combos, 0.0,
        resume_manifest, len(patch_trials),
    )
    log(f'[Canonical Phase C setup] C2 prep: kept {len(_kept_combos)} non-equal-k combos')
    if _indiv_df_in_mem is not None:
        _local_c1_partial = os.path.join(PHASE_30_DIR, 'individual_head_effects_partial.csv')
        _local_c1_checkpoint = os.path.join(PHASE_30_DIR, 'checkpoint_c1.json')
        _indiv_df_in_mem.to_csv(_local_c1_partial, index=False)
        _completed_heads = set()
        for _, _row in _indiv_df_in_mem.iterrows():
            _completed_heads.add((int(_row['layer']), int(_row['head'])))
        _save_c1_checkpoint(
            _local_c1_checkpoint, _completed_heads, 0.0,
            resume_manifest, len(patch_trials),
        )
        log(f'[Canonical Phase C setup] C1 prep: {len(_completed_heads)} heads marked completed')
    else:
        log(f'[Canonical Phase C setup] WARNING individual_head_effects[.csv|_partial.csv] missing on '
            f'Drive AND local; _run_c1 will recompute all heads')
    save_to_drive(PHASE_30_DIR, '30_patching')
    for _final in ['dose_response_v2.csv', 'individual_head_effects.csv']:
        for _base in [PHASE_30_DIR, PHASE_30_DRIVE]:
            _p = os.path.join(_base, _final)
            if os.path.exists(_p):
                try:
                    os.remove(_p)
                except OSError as _e:
                    log(f'[Canonical Phase C setup] WARNING failed to remove {_p}: {_e}')
    log(f'[Canonical Phase C setup] complete: phase_30_cached=False; '
        f'C1 resumes from completed_heads, C2 re-processes equal-k with k={_correct_k_at_eq}')
elif _saved_k_at_eq is not None:
    log(f'[Canonical Phase C setup] no update needed: saved k={_saved_k_at_eq} '
        f'matches correct k={_correct_k_at_eq}')

phase_30_cached = phase_done('30_patching', extra_required=expected_cfg_c) and all(
    os.path.exists(os.path.join(PHASE_30_DRIVE, fname)) for fname in PHASE_30_REQUIRED
)
phase_38_cached = phase_done('38_behavioral', extra_required=expected_cfg_gamma) and all(
    os.path.exists(os.path.join(PHASE_38_DRIVE, fname)) for fname in PHASE_38_REQUIRED
)

if phase_30_cached and phase_38_cached:
    log('[Phase C] cached on Drive -> loading 30_patching and 38_behavioral')
    _copy_dir(PHASE_30_DRIVE, PHASE_30_DIR)
    _copy_dir(PHASE_38_DRIVE, PHASE_38_DIR)
    individual_head_effects_df = pd.read_csv(os.path.join(PHASE_30_DIR, 'individual_head_effects.csv'))
    dose_response_v2_df = pd.read_csv(os.path.join(PHASE_30_DIR, 'dose_response_v2.csv'))
    flip_rates_df = pd.read_csv(os.path.join(PHASE_38_DIR, 'flip_rates.csv'))
else:
    clean_logits_dict = _load_main_logits_dict()
    source_tensor_cache = {
        tid: torch.as_tensor(arr, dtype=MODEL_TORCH_DTYPE)
        for tid, arr in main_acts.items()
    }
    clean_metric_cache = _cache_clean_metrics(
        patch_trials=patch_trials,
        prompt_by_tid=prompt_by_tid,
        clean_logits_dict=clean_logits_dict,
        model_obj=model,
    )

    phase30_reused = False
    if phase_30_cached:
        log('[Phase C] 30_patching cached; reusing C1/C2 and deriving γ only')
        _copy_dir(PHASE_30_DRIVE, PHASE_30_DIR)
        individual_head_effects_df = pd.read_csv(os.path.join(PHASE_30_DIR, 'individual_head_effects.csv'))
        dose_response_v2_df = pd.read_csv(os.path.join(PHASE_30_DIR, 'dose_response_v2.csv'))
        phase30_reused = True
    else:
        c1_elapsed_sec, c1_n_heads_processed = _run_c1(
            model_obj=model,
            patch_trials=patch_trials,
            prompt_by_tid=prompt_by_tid,
            source_tensor_cache=source_tensor_cache,
            clean_metric_cache=clean_metric_cache,
            total_heads=TOTAL_HEADS,
            head_dim=D,
            hidden_size=HIDDEN,
            model_dtype=MODEL_TORCH_DTYPE,
            resume_manifest=resume_manifest,
        )
        individual_head_effects_df = pd.read_csv(os.path.join(PHASE_30_DIR, 'individual_head_effects.csv'))
        expected_c1_rows = TOTAL_HEADS * len(patch_trials)
        if len(individual_head_effects_df) != expected_c1_rows:
            raise AssertionError(f'C1 row count mismatch: {len(individual_head_effects_df)} != {expected_c1_rows}')
        dose_response_v2_df, c2_elapsed_sec = _run_c2(
            model_obj=model,
            patch_trials=patch_trials,
            prompt_by_tid=prompt_by_tid,
            source_tensor_cache=source_tensor_cache,
            clean_metric_cache=clean_metric_cache,
            group_defs=group_defs,
            ratios_dose=RATIOS_DOSE,
            total_heads=TOTAL_HEADS,
            head_dim=D,
            hidden_size=HIDDEN,
            model_dtype=MODEL_TORCH_DTYPE,
            resume_manifest=resume_manifest,
            k_by_ratio=K_BY_RATIO,
        )
        expected_c2_rows = len(GROUP_NAMES) * len(RATIOS_DOSE) * len(patch_trials)
        if len(dose_response_v2_df) != expected_c2_rows:
            raise AssertionError(f'C2 row count mismatch: {len(dose_response_v2_df)} != {expected_c2_rows}')
        save_phase(PHASE_30_DIR, '30_patching', {
            **expected_cfg_c,
            'model_id': MODEL_ID,
            'c1_elapsed_sec': round(c1_elapsed_sec, 3),
            'c2_elapsed_sec': round(c2_elapsed_sec, 3),
            'c1_n_heads_processed': int(c1_n_heads_processed),
            'c2_n_runs': int(len(dose_response_v2_df)),
            'cell_counts': cell_counts,
        })
        if globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
            save_to_drive(PHASE_30_DIR, '30_patching')

    flip_rates_df, flip_agg = _derive_gamma(dose_response_v2_df.copy())
    save_rows_csv(
        flip_rates_df.to_dict('records'),
        os.path.join(PHASE_38_DIR, 'flip_rates.csv'),
        'flip_rates.csv',
        columns=GAMMA_FIELDNAMES,
    )
    save_phase(PHASE_38_DIR, '38_behavioral', {
        **expected_cfg_gamma,
        'model_id': MODEL_ID,
        'flip_rates_aggregated': flip_agg,
    })
    if phase30_reused and globals().get('DRIVE_SAVE', False) and 'save_to_drive' in globals():
        save_to_drive(PHASE_38_DIR, '38_behavioral')

globals()['individual_head_effects_df'] = individual_head_effects_df
globals()['dose_response_v2_df'] = dose_response_v2_df
globals()['flip_rates_df'] = flip_rates_df
globals()['RATIOS_DOSE_PHASE_C'] = RATIOS_DOSE
globals()['RATIOS_BEHAVIORAL_PHASE_C'] = RATIOS_BEHAVIORAL

_log_mem('Phase C end')


In [ ]:
# ============================================================
# Cell 10: Phase D — Readout Decomposition
# Purpose:
#   Derive residual-space readout decomposition for canonical head groups
#   using the same Pattern A input-overwrite mechanism as Phase C.2.
#
# Inputs expected in scope:
#   model, tokenizer, arch, L, H, D, HIDDEN,
#   main_acts, main_logits, main_rows, _td_df, _ad_df,
#   rsa_per_head_df, perturbation_per_head_df, cell_classification_df,
#   individual_head_effects_df, dose_response_v2_df, flip_rates_df,
#   log, ensure_dir, save_json, save_rows_csv, save_phase, phase_done,
#   load_model, unload_model,
#   PIPELINE_VERSION, SEED, PERM_SEED, MODEL_ID, MODEL_SHORT, MODEL_DTYPE,
#   LOCAL_BASE, DRIVE_BASE
#
# Spec refs:
#   Appendix A.5.3 group patching
#   Appendix A.6 readout metric
# ============================================================

import gc
import math
import os
import shutil
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F  # noqa: F401; used by _apply_output_delta from Phase C


PHASE_40_DIR = ensure_dir(os.path.join(LOCAL_BASE, '40_readout'))
PHASE_40_DRIVE = os.path.join(DRIVE_BASE, '40_readout')
PHASE_40_REQUIRED = [
    'group_readout.csv',
    'group_readout_clean.csv',
    'perturbation_vectors.npz',
    'config.json',
]

CREL_PAIRS_PHASE_D = [
    ('SAME', 'OPP'),
    ('SAME', 'MORE'),
    ('SAME', 'LESS'),
    ('OPP', 'MORE'),
    ('OPP', 'LESS'),
    ('MORE', 'LESS'),
]
GROUPS_READOUT = [
    'hihp_imp_desc',
    'hilp_imp_desc',
    'C_imp_asc',
    'D_imp_asc',
    'std_imp_asc',
]
RATIOS_READOUT = [0.05, 0.10, 0.15]
CACHE_LAYER_INDICES = list(range(max(0, L - 8), L))
GROUP_READOUT_COLUMNS = [
    'group',
    'ratio',
    'cache_layer',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'readout_comp_signed',
    'perturbation_norm',
    'ortho_norm',
]


def _copy_tree(src_dir, dst_dir):
    if not os.path.isdir(src_dir):
        return
    ensure_dir(dst_dir)
    for root, _, files_in_root in os.walk(src_dir):
        rel_root = os.path.relpath(root, src_dir)
        dst_root = dst_dir if rel_root == '.' else os.path.join(dst_dir, rel_root)
        ensure_dir(dst_root)
        for fname in files_in_root:
            shutil.copy2(os.path.join(root, fname), os.path.join(dst_root, fname))


def _norm_pair_id_local(x):
    if '_norm_pair_id' in globals():
        return _norm_pair_id(x)
    try:
        return int(x)
    except Exception:
        s = str(x)
        if s.startswith('pair') and s[4:].isdigit():
            return int(s[4:])
        return s


def _pair_sort_key_local(x):
    if '_pair_sort_key' in globals():
        return _pair_sort_key(x)
    try:
        return (0, int(x))
    except Exception:
        s = str(x)
        if s.startswith('pair') and s[4:].isdigit():
            return (0, int(s[4:]))
        return (1, s)


def _to_int_or_none(x):
    if x is None or (isinstance(x, float) and np.isnan(x)) or pd.isna(x):
        return None
    return int(x)


def _resolve_tensor(saved_obj):
    return getattr(saved_obj, 'value', saved_obj)


def _torch_dtype_local(dtype_name):
    if '_torch_dtype' in globals():
        return _torch_dtype(dtype_name)
    if 'get_torch_dtype' in globals():
        return get_torch_dtype(dtype_name)
    if dtype_name == 'float32':
        return torch.float32
    if dtype_name == 'float16':
        return torch.float16
    return torch.bfloat16


def _extract_unembedding_matrix(model_obj):
    candidates = []
    if hasattr(model_obj, 'lm_head') and hasattr(model_obj.lm_head, 'weight'):
        candidates.append(model_obj.lm_head.weight)
    if hasattr(model_obj, 'model') and hasattr(model_obj.model, 'lm_head') and hasattr(model_obj.model.lm_head, 'weight'):
        candidates.append(model_obj.model.lm_head.weight)
    for weight in candidates:
        try:
            return weight.detach().float().cpu().numpy()
        except Exception:
            pass
    with model_obj.trace('A:'):
        weight_proxy = model_obj.lm_head.weight.save()
    return _resolve_tensor(weight_proxy).detach().float().cpu().numpy()


def _trace_clean_resid(model_obj, prompt, cache_layer_indices):
    resid_saves = {}
    with model_obj.trace(prompt):
        for cache_layer in sorted(cache_layer_indices):
            resid_saves[int(cache_layer)] = model_obj.model.layers[int(cache_layer)].output[0, -1, :].save()
        logits_proxy = model_obj.output.logits[0, -1, :].save()
    logits_np = _resolve_tensor(logits_proxy).detach().float().cpu().numpy()
    resid_dict = {}
    for cache_layer, saved_proxy in resid_saves.items():
        resid_dict[int(cache_layer)] = _resolve_tensor(saved_proxy).detach().float().cpu().numpy()
    return logits_np, resid_dict


def _trace_group_patch_resid(model_obj, prompt, source_tensor, group_heads, cache_layer_indices, head_dim, hidden_size, model_dtype):
    resid_saves = {}
    patches_by_layer = defaultdict(list)
    for layer_idx, head_idx in sorted(group_heads, key=lambda lh: (int(lh[0]), int(lh[1]))):
        patches_by_layer[int(layer_idx)].append(int(head_idx))
    cache_set = set(int(x) for x in cache_layer_indices)
    with model_obj.trace(prompt):
        for layer_idx in range(int(L)):
            for head_idx in patches_by_layer.get(layer_idx, []):
                _apply_output_delta(
                    model_obj.model.layers[layer_idx],
                    int(head_idx),
                    source_tensor[layer_idx, head_idx],
                    head_dim,
                    hidden_size,
                    model_dtype,
                )
            if layer_idx in cache_set:
                resid_saves[layer_idx] = model_obj.model.layers[layer_idx].output[0, -1, :].save()
        logits_proxy = model_obj.output.logits[0, -1, :].save()
    logits_np = _resolve_tensor(logits_proxy).detach().float().cpu().numpy()
    resid_dict = {}
    for cache_layer, saved_proxy in resid_saves.items():
        resid_dict[int(cache_layer)] = _resolve_tensor(saved_proxy).detach().float().cpu().numpy()
    return logits_np, resid_dict


def _build_group_orderings_local():
    if '_build_group_orderings' in globals():
        _, group_defs_local, cell_counts_local = _build_group_orderings(
            cell_classification_df.copy(),
            rsa_per_head_df.copy(),
            perturbation_per_head_df.copy(),
        )
        return group_defs_local, cell_counts_local

    merged = (
        cell_classification_df[['layer', 'head', 'cell']]
        .merge(rsa_per_head_df[['layer', 'head', 'rsa_max']], on=['layer', 'head'], how='left')
        .merge(perturbation_per_head_df[['layer', 'head', 'perturbation_L2']], on=['layer', 'head'], how='left')
        .copy()
    )
    merged['layer'] = merged['layer'].astype(int)
    merged['head'] = merged['head'].astype(int)
    merged['rsa_max'] = merged['rsa_max'].astype(float)
    merged['perturbation_L2'] = merged['perturbation_L2'].astype(float)

    def _heads(df_slice, cols, ascending):
        ordered = df_slice.sort_values(cols, ascending=ascending, kind='mergesort')
        return list(zip(ordered['layer'].astype(int), ordered['head'].astype(int)))

    group_defs_local = {
        'hihp_imp_desc': _heads(merged[merged['cell'] == 'hihp'], ['rsa_max', 'layer', 'head'], [False, True, True]),
        'hilp_imp_desc': _heads(merged[merged['cell'] == 'hilp'], ['rsa_max', 'layer', 'head'], [False, True, True]),
        'C_imp_asc': _heads(merged[merged['cell'] == 'C'], ['rsa_max', 'layer', 'head'], [True, True, True]),
        'D_imp_asc': _heads(merged[merged['cell'] == 'D'], ['rsa_max', 'layer', 'head'], [True, True, True]),
        'std_imp_asc': _heads(merged, ['rsa_max', 'layer', 'head'], [True, True, True]),
    }
    cell_counts_local = {
        cell: int((merged['cell'] == cell).sum())
        for cell in ['hihp', 'hilp', 'C', 'D']
    }
    return group_defs_local, cell_counts_local


def _build_phase_d_trials(trials_df, diag_df, W_U_local):
    if 'prompt' not in trials_df.columns:
        raise KeyError('_td_df must contain prompt')
    trial_rows = trials_df.to_dict('records')
    diag_lookup = {row['trial_id']: row for row in diag_df.to_dict('records')}
    trial_by_key = {}
    prompt_by_tid = {}
    for row in trial_rows:
        key = (row['crel'], _norm_pair_id_local(row['pair_id']), row['attr_dim'])
        trial_by_key[key] = row
        prompt_by_tid[row['trial_id']] = row['prompt']

    pair_ids_sorted = sorted(trials_df['pair_id'].unique().tolist(), key=_pair_sort_key_local)
    attr_dims_sorted = sorted(trials_df['attr_dim'].unique().tolist())
    patch_trials = []
    for c1, c2 in CREL_PAIRS_PHASE_D:
        for pair_id_raw in pair_ids_sorted:
            pair_id_norm = _norm_pair_id_local(pair_id_raw)
            for attr_dim in attr_dims_sorted:
                tgt_row = trial_by_key.get((c1, pair_id_norm, attr_dim))
                src_row = trial_by_key.get((c2, pair_id_norm, attr_dim))
                if tgt_row is None or src_row is None:
                    raise KeyError(f'Missing alpha-rule trial for {(c1, c2, pair_id_raw, attr_dim)}')
                diag_row = diag_lookup.get(tgt_row['trial_id'])
                if diag_row is None:
                    raise KeyError(f'Missing _ad_df row for trial_id={tgt_row["trial_id"]}')
                correct_tok = _to_int_or_none(diag_row.get('correct_first_subtoken_id'))
                incorrect_tok = _to_int_or_none(diag_row.get('incorrect_first_subtoken_id'))
                readout_dir = None
                if correct_tok is None or incorrect_tok is None:
                    log(
                        f'WARNING Phase D readout_dir missing token ids for '
                        f'trial_id={tgt_row["trial_id"]} crel_pair={c1}-{c2}'
                    )
                else:
                    direction = W_U_local[correct_tok] - W_U_local[incorrect_tok]
                    denom = float(np.linalg.norm(direction))
                    if denom <= 1e-10:
                        log(
                            f'WARNING Phase D readout_dir near-zero norm for '
                            f'trial_id={tgt_row["trial_id"]} tokens=({correct_tok},{incorrect_tok})'
                        )
                    else:
                        readout_dir = direction / denom
                patch_trials.append({
                    'crel_pair': f'{c1}-{c2}',
                    'pair_id': tgt_row['pair_id'],
                    'attr_dim': attr_dim,
                    'tgt_tid': tgt_row['trial_id'],
                    'src_tid': src_row['trial_id'],
                    'readout_dir': readout_dir,
                })
    if len(patch_trials) != 120:
        raise AssertionError(f'Expected 120 Phase D tuples, got {len(patch_trials)}')
    return patch_trials, prompt_by_tid


MODEL_TORCH_DTYPE = _torch_dtype_local(MODEL_DTYPE)
TOTAL_HEADS = int(L * H)
group_orderings, cell_counts = _build_group_orderings_local()
expected_cfg_d = {
    'phase_name': '40_readout',
    'source_rule': 'alpha_same_pair_same_attr_cross_crel',
    'patch_mechanism': 'input_overwrite_v5_1_pattern',
    'patch_mechanism_version': 'v2_canonical_patch_mechanism',
    'ratios_readout': RATIOS_READOUT,
    'cache_layer_indices': list(CACHE_LAYER_INDICES),
    'groups': GROUPS_READOUT,
    'readout_direction_source': 'W_U_correct_minus_incorrect_normalized',
}

phase_40_cached = phase_done('40_readout', extra_required=expected_cfg_d) and all(
    os.path.exists(os.path.join(PHASE_40_DRIVE, fname)) for fname in PHASE_40_REQUIRED
)

if phase_40_cached:
    log('[Phase D] cached on Drive -> loading 40_readout')
    _copy_tree(PHASE_40_DRIVE, PHASE_40_DIR)
    group_readout_df = pd.read_csv(os.path.join(PHASE_40_DIR, 'group_readout.csv'))
    group_readout_clean_df = pd.read_csv(os.path.join(PHASE_40_DIR, 'group_readout_clean.csv'))
else:
    if 'model' not in globals() or model is None:
        log('[Phase D] model not in globals -> loading')
        model, tokenizer, arch = load_model(MODEL_ID, MODEL_DTYPE)
    elif 'tokenizer' not in globals() or tokenizer is None:
        tokenizer = getattr(model, 'tokenizer', None)
    if tokenizer is None:
        raise RuntimeError('Tokenizer unavailable for Phase D')

    for layer_idx in range(int(L)):
        proj = model.model.layers[layer_idx].self_attn.o_proj
        assert getattr(proj, 'bias', None) is None, f'o_proj.bias must be None at layer {layer_idx}'

    W_U = _extract_unembedding_matrix(model)
    patch_trials, prompt_by_tid = _build_phase_d_trials(_td_df.copy(), _ad_df.copy(), W_U)
    source_tensor_cache = {
        tid: torch.as_tensor(arr, dtype=MODEL_TORCH_DTYPE)
        for tid, arr in main_acts.items()
    }

    log(f'[Phase D] caching clean residuals for {len(set(pt["tgt_tid"] for pt in patch_trials))} target trials')
    clean_resid_by_tid = {}
    for idx, tgt_tid in enumerate(sorted(set(pt['tgt_tid'] for pt in patch_trials)), start=1):
        _, resid_dict = _trace_clean_resid(model, prompt_by_tid[tgt_tid], CACHE_LAYER_INDICES)
        clean_resid_by_tid[tgt_tid] = resid_dict
        if idx % 20 == 0 or idx == len(set(pt['tgt_tid'] for pt in patch_trials)):
            log(f'[Phase D] clean residual cache {idx}/{len(set(pt["tgt_tid"] for pt in patch_trials))}')

    group_readout_rows = []
    perturbation_vectors = {}
    t0_phase_d = time.time()
    combo_total = len(GROUPS_READOUT) * len(RATIOS_READOUT)
    combo_done = 0

    for group_name in GROUPS_READOUT:
        ordered_heads = list(group_orderings.get(group_name, []))
        for ratio in RATIOS_READOUT:
            k = max(1, int(float(ratio) * TOTAL_HEADS))
            k_actual = min(k, len(ordered_heads))
            active_heads = ordered_heads[:k_actual]
            combo_done += 1
            if k_actual == 0:
                log(f'WARNING [Phase D] group={group_name} ratio={ratio:.2f}: no available heads')
            else:
                log(
                    f'[Phase D] group={group_name} ratio={ratio:.2f} '
                    f'k={k} k_actual={k_actual} ({combo_done}/{combo_total})'
                )
            for pt in patch_trials:
                try:
                    if k_actual == 0:
                        raise RuntimeError(f'group {group_name} has zero available heads')
                    source_tensor = source_tensor_cache.get(pt['src_tid'])
                    if source_tensor is None:
                        raise KeyError(f'main_acts missing source trial {pt["src_tid"]}')
                    _, patched_resid = _trace_group_patch_resid(
                        model_obj=model,
                        prompt=prompt_by_tid[pt['tgt_tid']],
                        source_tensor=source_tensor,
                        group_heads=active_heads,
                        cache_layer_indices=CACHE_LAYER_INDICES,
                        head_dim=int(D),
                        hidden_size=int(HIDDEN),
                        model_dtype=MODEL_TORCH_DTYPE,
                    )
                except Exception as exc:
                    log(
                        f'WARNING [Phase D] patch failed group={group_name} ratio={ratio:.2f} '
                        f'trial={pt["tgt_tid"]}: {type(exc).__name__}: {str(exc)[:200]}'
                    )
                    patched_resid = {}
                    clean_resid = clean_resid_by_tid.get(pt['tgt_tid'], {})
                    for cache_layer in CACHE_LAYER_INDICES:
                        if int(cache_layer) not in clean_resid:
                            log(
                                f'WARNING [Phase D] missing clean resid after exception '
                                f'group={group_name} ratio={ratio:.2f} trial={pt["tgt_tid"]} cache_layer={cache_layer}'
                            )
                        group_readout_rows.append({
                            'group': group_name,
                            'ratio': float(ratio),
                            'cache_layer': int(cache_layer),
                            'crel_pair': pt['crel_pair'],
                            'pair_id': pt['pair_id'],
                            'attr_dim': pt['attr_dim'],
                            'readout_comp_signed': float('nan'),
                            'perturbation_norm': float('nan'),
                            'ortho_norm': float('nan'),
                        })
                    continue

                clean_resid = clean_resid_by_tid.get(pt['tgt_tid'], {})
                for cache_layer in CACHE_LAYER_INDICES:
                    clean_vec = clean_resid.get(int(cache_layer))
                    patched_vec = patched_resid.get(int(cache_layer))
                    if clean_vec is None or patched_vec is None:
                        if clean_vec is None:
                            log(
                                f'WARNING [Phase D] missing clean resid '
                                f'group={group_name} ratio={ratio:.2f} trial={pt["tgt_tid"]} cache_layer={cache_layer}'
                            )
                        if patched_vec is None:
                            log(
                                f'WARNING [Phase D] missing patched resid '
                                f'group={group_name} ratio={ratio:.2f} trial={pt["tgt_tid"]} cache_layer={cache_layer}'
                            )
                        group_readout_rows.append({
                            'group': group_name,
                            'ratio': float(ratio),
                            'cache_layer': int(cache_layer),
                            'crel_pair': pt['crel_pair'],
                            'pair_id': pt['pair_id'],
                            'attr_dim': pt['attr_dim'],
                            'readout_comp_signed': float('nan'),
                            'perturbation_norm': float('nan'),
                            'ortho_norm': float('nan'),
                        })
                        continue

                    pv = np.asarray(patched_vec, dtype=np.float32) - np.asarray(clean_vec, dtype=np.float32)
                    perturbation_norm = float(np.linalg.norm(pv))
                    readout_dir = pt['readout_dir']
                    if readout_dir is None:
                        readout_comp_signed = float('nan')
                        ortho_norm = float('nan')
                    else:
                        readout_comp_signed = float(np.dot(pv, readout_dir))
                        ortho_norm = float(np.linalg.norm(pv - (readout_comp_signed * readout_dir)))

                    group_readout_rows.append({
                        'group': group_name,
                        'ratio': float(ratio),
                        'cache_layer': int(cache_layer),
                        'crel_pair': pt['crel_pair'],
                        'pair_id': pt['pair_id'],
                        'attr_dim': pt['attr_dim'],
                        'readout_comp_signed': readout_comp_signed,
                        'perturbation_norm': perturbation_norm,
                        'ortho_norm': ortho_norm,
                    })

                    ratio_str = f'{float(ratio):.2f}'
                    pv_key = f'{group_name}_{ratio_str}_{int(cache_layer)}_{pt["crel_pair"]}_{pt["pair_id"]}_{pt["attr_dim"]}'
                    perturbation_vectors[pv_key] = pv.astype(np.float32, copy=False)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    elapsed_sec = time.time() - t0_phase_d
    group_readout_df = save_rows_csv(
        group_readout_rows,
        os.path.join(PHASE_40_DIR, 'group_readout.csv'),
        'group_readout.csv',
        columns=GROUP_READOUT_COLUMNS,
    )
    group_readout_clean_df = save_rows_csv(
        [
            row for row in group_readout_rows
            if str(row['crel_pair']) in {'SAME-OPP', 'MORE-LESS'}
        ],
        os.path.join(PHASE_40_DIR, 'group_readout_clean.csv'),
        'group_readout_clean.csv',
        columns=GROUP_READOUT_COLUMNS,
    )
    np.savez_compressed(os.path.join(PHASE_40_DIR, 'perturbation_vectors.npz'), **perturbation_vectors)
    log(
        f'Saved perturbation_vectors.npz: entries={len(perturbation_vectors)} '
        f'path={os.path.join(PHASE_40_DIR, "perturbation_vectors.npz")}'
    )
    save_phase(PHASE_40_DIR, '40_readout', {
        **expected_cfg_d,
        'n_readout_rows': int(len(group_readout_df)),
        'elapsed_sec': round(float(elapsed_sec), 3),
        'cache_layer_count': int(len(CACHE_LAYER_INDICES)),
    })

globals()['group_readout_df'] = group_readout_df
globals()['group_readout_clean_df'] = group_readout_clean_df
globals()['RATIOS_READOUT'] = RATIOS_READOUT
globals()['CACHE_LAYER_INDICES'] = CACHE_LAYER_INDICES

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# Cell 11: Phase 38 — Behavioral (derivation only)
# Purpose:
#   Derive behavioral accuracy / flip metrics directly from Phase C.2
#   dose_response_v2.csv outputs, with no new forward passes.
#
# Inputs expected in scope:
#   dose_response_v2_df, _td_df, _ad_df,
#   log, ensure_dir, save_json, save_rows_csv, save_phase, phase_done,
#   PIPELINE_VERSION, MODEL_ID, MODEL_SHORT, LOCAL_BASE, DRIVE_BASE
#
# Spec refs:
#   Appendix A.6 readout metric
#
# Canonical review-time implementation notes:
#   - Pure derivation only: no new model traces / forward passes.
#   - Missing argmax values remain NaN; no `-1` top1 sentinel.
# ============================================================

import os
import shutil
import time

import numpy as np
import pandas as pd


PHASE_38_DIR = ensure_dir(os.path.join(LOCAL_BASE, '38_behavioral'))
PHASE_38_DRIVE = os.path.join(DRIVE_BASE, '38_behavioral')
PHASE_38_REQUIRED = [
    'behavioral_per_trial.csv',
    'behavioral_aggregate.csv',
    'flip_rates.csv',
    'config.json',
]
RATIOS_BEHAVIORAL = [0.05, 0.10, 0.20, 0.30, 0.50]
CELLS_BEHAVIORAL = ['hihp_imp_desc', 'hilp_imp_desc', 'C_imp_asc', 'D_imp_asc']
BEHAVIORAL_PER_TRIAL_COLUMNS = [
    'group',
    'ratio',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'tgt_tok',
    'argmax_clean',
    'argmax_patched',
    'is_correct_clean',
    'is_correct_patched',
    'is_argmax_flip',
    'target_logit_delta_signed',
    'source_logit_delta_signed',
]
BEHAVIORAL_AGG_COLUMNS = [
    'group',
    'ratio',
    'n_trials',
    'n_valid',
    'accuracy_clean',
    'accuracy_patched',
    'accuracy_degradation',
    'flip_rate',
    'target_logit_delta_mean',
    'source_logit_delta_mean',
]
FLIP_RATE_COLUMNS = [
    'group',
    'ratio',
    'crel_pair',
    'pair_id',
    'attr_dim',
    'is_argmax_flip',
]


def _copy_tree(src_dir, dst_dir):
    if not os.path.isdir(src_dir):
        return
    ensure_dir(dst_dir)
    for root, _, files_in_root in os.walk(src_dir):
        rel_root = os.path.relpath(root, src_dir)
        dst_root = dst_dir if rel_root == '.' else os.path.join(dst_dir, rel_root)
        ensure_dir(dst_root)
        for fname in files_in_root:
            shutil.copy2(os.path.join(root, fname), os.path.join(dst_root, fname))


def _norm_pair_id_local(x):
    if '_norm_pair_id' in globals():
        return _norm_pair_id(x)
    try:
        return int(x)
    except Exception:
        s = str(x)
        if s.startswith('pair') and s[4:].isdigit():
            return int(s[4:])
        return s


def _as_int_or_nan(x):
    if x is None or (isinstance(x, float) and np.isnan(x)) or pd.isna(x):
        return np.nan
    return int(x)


def _binary_metric(value):
    if value is None:
        return np.nan
    return int(value)


expected_cfg_38 = {
    'phase_name': '38_behavioral',
    'derived_from': 'dose_response_v2.csv',
    'ratios_behavioral': RATIOS_BEHAVIORAL,
    'cells_behavioral': CELLS_BEHAVIORAL,
}

phase_38_cached = phase_done('38_behavioral', extra_required=expected_cfg_38) and all(
    os.path.exists(os.path.join(PHASE_38_DRIVE, fname)) for fname in PHASE_38_REQUIRED
)

if phase_38_cached:
    log('[Phase 38] cached on Drive -> loading 38_behavioral')
    _copy_tree(PHASE_38_DRIVE, PHASE_38_DIR)
    behavioral_per_trial_df = pd.read_csv(os.path.join(PHASE_38_DIR, 'behavioral_per_trial.csv'))
    behavioral_aggregate_df = pd.read_csv(os.path.join(PHASE_38_DIR, 'behavioral_aggregate.csv'))
    flip_rates_df = pd.read_csv(os.path.join(PHASE_38_DIR, 'flip_rates.csv'))
else:
    if 'dose_response_v2_df' not in globals() or dose_response_v2_df is None:
        local_dose = os.path.join(LOCAL_BASE, '30_patching', 'dose_response_v2.csv')
        drive_dose = os.path.join(DRIVE_BASE, '30_patching', 'dose_response_v2.csv')
        if os.path.exists(local_dose):
            dose_response_v2_df = pd.read_csv(local_dose)
        elif os.path.exists(drive_dose):
            dose_response_v2_df = pd.read_csv(drive_dose)
        else:
            raise FileNotFoundError('dose_response_v2_df is not in scope and dose_response_v2.csv was not found')

    t0_phase_38 = time.time()
    ratio_keep = {round(float(r), 6) for r in RATIOS_BEHAVIORAL}
    kept_df = dose_response_v2_df.copy()
    kept_df['ratio'] = kept_df['ratio'].astype(float)
    kept_df = kept_df[
        kept_df['group'].isin(CELLS_BEHAVIORAL)
        & kept_df['ratio'].round(6).isin(ratio_keep)
    ].copy()

    trial_lookup = {}
    for row in _td_df.to_dict('records'):
        trial_lookup[(row['crel'], _norm_pair_id_local(row['pair_id']), row['attr_dim'])] = row['trial_id']
    diag_lookup = {row['trial_id']: row for row in _ad_df.to_dict('records')}

    target_tok_lookup = {}
    missing_target_keys = set()
    for row in kept_df[['crel_pair', 'pair_id', 'attr_dim']].drop_duplicates().to_dict('records'):
        crel_pair = str(row['crel_pair'])
        if '-' not in crel_pair:
            missing_target_keys.add((crel_pair, row['pair_id'], row['attr_dim']))
            target_tok_lookup[(crel_pair, row['pair_id'], row['attr_dim'])] = None
            continue
        c1, _ = crel_pair.split('-', 1)
        trial_id = trial_lookup.get((c1, _norm_pair_id_local(row['pair_id']), row['attr_dim']))
        diag_row = diag_lookup.get(trial_id) if trial_id is not None else None
        tgt_tok = None
        if diag_row is not None and pd.notna(diag_row.get('correct_first_subtoken_id')):
            tgt_tok = int(diag_row['correct_first_subtoken_id'])
        else:
            missing_target_keys.add((crel_pair, row['pair_id'], row['attr_dim']))
        target_tok_lookup[(crel_pair, row['pair_id'], row['attr_dim'])] = tgt_tok

    for crel_pair, pair_id, attr_dim in sorted(missing_target_keys):
        log(
            f'WARNING [Phase 38] missing tgt_tok lookup for '
            f'crel_pair={crel_pair} pair_id={pair_id} attr_dim={attr_dim}'
        )

    per_trial_rows = []
    for row in kept_df.to_dict('records'):
        key = (row['crel_pair'], row['pair_id'], row['attr_dim'])
        tgt_tok = target_tok_lookup.get(key)
        argmax_clean = _as_int_or_nan(row.get('argmax_token_id_clean'))
        argmax_patched = _as_int_or_nan(row.get('argmax_token_id_patched'))

        if tgt_tok is None or pd.isna(argmax_clean):
            is_correct_clean = np.nan
        else:
            is_correct_clean = int(int(argmax_clean) == int(tgt_tok))

        if tgt_tok is None or pd.isna(argmax_patched):
            is_correct_patched = np.nan
        else:
            is_correct_patched = int(int(argmax_patched) == int(tgt_tok))

        if pd.isna(argmax_clean) or pd.isna(argmax_patched):
            is_argmax_flip = np.nan
        else:
            is_argmax_flip = int(int(argmax_clean) != int(argmax_patched))

        per_trial_rows.append({
            'group': row['group'],
            'ratio': float(row['ratio']),
            'crel_pair': row['crel_pair'],
            'pair_id': row['pair_id'],
            'attr_dim': row['attr_dim'],
            'tgt_tok': tgt_tok if tgt_tok is not None else np.nan,
            'argmax_clean': argmax_clean,
            'argmax_patched': argmax_patched,
            'is_correct_clean': is_correct_clean,
            'is_correct_patched': is_correct_patched,
            'is_argmax_flip': is_argmax_flip,
            'target_logit_delta_signed': row.get('delta_target_logit_signed', np.nan),
            'source_logit_delta_signed': row.get('delta_source_logit_signed', np.nan),
        })

    behavioral_per_trial_df = save_rows_csv(
        per_trial_rows,
        os.path.join(PHASE_38_DIR, 'behavioral_per_trial.csv'),
        'behavioral_per_trial.csv',
        columns=BEHAVIORAL_PER_TRIAL_COLUMNS,
    )

    agg_rows = []
    accuracy_aggregated = {}
    if len(behavioral_per_trial_df) > 0:
        grouped = behavioral_per_trial_df.groupby(['group', 'ratio'], sort=True, dropna=False)
        for (group_name, ratio), df_gr in grouped:
            clean_series = pd.to_numeric(df_gr['is_correct_clean'], errors='coerce')
            patched_series = pd.to_numeric(df_gr['is_correct_patched'], errors='coerce')
            flip_series = pd.to_numeric(df_gr['is_argmax_flip'], errors='coerce')
            target_delta_series = pd.to_numeric(df_gr['target_logit_delta_signed'], errors='coerce')
            source_delta_series = pd.to_numeric(df_gr['source_logit_delta_signed'], errors='coerce')

            accuracy_clean = float(clean_series.mean()) if clean_series.notna().any() else np.nan
            accuracy_patched = float(patched_series.mean()) if patched_series.notna().any() else np.nan
            accuracy_degradation = (
                float(accuracy_clean - accuracy_patched)
                if np.isfinite(accuracy_clean) and np.isfinite(accuracy_patched)
                else np.nan
            )
            flip_rate = float(flip_series.mean()) if flip_series.notna().any() else np.nan
            target_delta_mean = float(target_delta_series.mean()) if target_delta_series.notna().any() else np.nan
            source_delta_mean = float(source_delta_series.mean()) if source_delta_series.notna().any() else np.nan

            agg_rows.append({
                'group': group_name,
                'ratio': float(ratio),
                'n_trials': int(len(df_gr)),
                'n_valid': int(patched_series.notna().sum()),
                'accuracy_clean': accuracy_clean,
                'accuracy_patched': accuracy_patched,
                'accuracy_degradation': accuracy_degradation,
                'flip_rate': flip_rate,
                'target_logit_delta_mean': target_delta_mean,
                'source_logit_delta_mean': source_delta_mean,
            })
            accuracy_aggregated.setdefault(group_name, {})[f'{float(ratio):.6f}'] = (
                None if not np.isfinite(accuracy_patched) else float(accuracy_patched)
            )

    behavioral_aggregate_df = save_rows_csv(
        agg_rows,
        os.path.join(PHASE_38_DIR, 'behavioral_aggregate.csv'),
        'behavioral_aggregate.csv',
        columns=BEHAVIORAL_AGG_COLUMNS,
    )

    flip_rates_df = save_rows_csv(
        behavioral_per_trial_df[FLIP_RATE_COLUMNS].to_dict('records'),
        os.path.join(PHASE_38_DIR, 'flip_rates.csv'),
        'flip_rates.csv',
        columns=FLIP_RATE_COLUMNS,
    )

    save_phase(PHASE_38_DIR, '38_behavioral', {
        **expected_cfg_38,
        'n_trials': int(len(behavioral_per_trial_df)),
        'accuracy_aggregated': accuracy_aggregated,
        'elapsed_sec': round(float(time.time() - t0_phase_38), 3),
    })

globals()['behavioral_per_trial_df'] = behavioral_per_trial_df
globals()['behavioral_aggregate_df'] = behavioral_aggregate_df
globals()['flip_rates_df'] = flip_rates_df
globals()['RATIOS_BEHAVIORAL'] = RATIOS_BEHAVIORAL


In [ ]:
# ============================================================
# Cell 7: Release Colab runtime (free compute units)
# ============================================================
from google.colab import runtime
log('Releasing Colab runtime (compute units stop billing).')
runtime.unassign()
